# Copilot Audit Log — Direct Ingester (Fabric)

End-to-end ingestion notebook that **calls Microsoft Graph directly** from inside Fabric, parses the audit-log records on the fly, and writes a Delta table the **AI-in-One Dashboard** and **ValueLens** PBITs consume directly.

**No PowerShell, no intermediate CSVs, no separate parser.** Replaces the previous flow:

```
OLD:  PowerShell script → local CSV → manual move into Files/audit_raw/ → Copilot_Audit_Log_Parser.ipynb → Delta
NEW:  This notebook (Graph → parse → Delta)
```

**Output**: Lakehouse Delta table `Copilot_Interactions_Parsed` (same name + schema as the existing parser, so the PBIT works without changes), and `audit_source_status`: one row per source the run tried (`Source`, `Status` = `ok`, `empty`, `skipped`, `forbidden` or `error`), which `AnalyticsHub_Load_Status` reads to flag a run that couldn't read a source.

**Schedule**: weekly (matches Microsoft Graph's 7-day audit-log query window). Use the **Schedule** button at the top of the notebook.

**Permissions**: app registration with `AuditLogsQuery.Read.All` (Application permission, admin-consented). No `Sites.Selected` needed — this path doesn't touch SharePoint.

**Faster sources (`AUDIT_SOURCE`)**: Graph audit search is slow on a large tenant. With `AUDIT_SOURCE = 'auto'` the
notebook reads each window from the fastest source that holds it, and writes the same rows either way:

| Source | Holds | Needs |
|---|---|---|
| Microsoft Sentinel `CopilotActivity` table | from when the Microsoft Copilot data connector was connected, for the workspace's retention | `ThreatHunting.Read.All` (Graph), or the Log Analytics Reader role with `SENTINEL_QUERY_VIA = 'loganalytics'` |
| Office 365 Management Activity API | the last 7 days | `ActivityFeed.Read` (Office 365 Management APIs) |
| Graph audit search | any date; only when `GRAPH_FOR_HISTORY = True` (a one-time history load) | `AuditLogsQuery.Read.All` |

Records are keyed by the audit record Id, so a window read from two sources, or a switch of source, never duplicates a row.
Without `GRAPH_FOR_HISTORY`, windows older than the sources hold are left out (a first load covers the last 7 days), and an
overwrite becomes a merge so the history already in the table is kept.


## 1. Configuration

Fill in the four config values below. The client secret should ideally come from a Key Vault — see the commented alternative.

**Lakehouse Schemas note**: the default `OUTPUT_TABLE` is `dbo.Copilot_Interactions_Parsed`. This works for both schema-enabled Lakehouses (the new Fabric default) and non-schema Lakehouses — `dbo` is the implicit default schema either way. If your Lakehouse uses a different schema, change the prefix.

**Audit source**: `AUDIT_SOURCE = 'feed'` (the default) reads the Office 365 Management Activity API, which holds
the last 7 days. `'auto'` reads Sentinel's `CopilotActivity` table for the hours it holds, otherwise the Management
Activity API. Older windows are left out with a note, so a first load covers 7 days; `GRAPH_FOR_HISTORY = True` reads
them from Graph audit search for one run. `'graph'` reads every window from Graph audit search (slow on a large
tenant), and `'sentinel'` reads only Sentinel.


In [ ]:
# === CONFIG ===
TENANT_ID     = '<your-tenant-guid>'                  # Entra -> Overview -> Tenant ID
CLIENT_ID     = '<your-app-reg-client-id>'            # Entra -> App registrations -> your app -> Application (client) ID
CLIENT_SECRET = '<your-client-secret-value>'          # The secret VALUE (not Secret ID). Prefer Key Vault - see below.

# --- Load mode ---------------------------------------------------------------------
# 'backfill'    = reload BACKFILL_DAYS of history: the initial load, or a reload. Rows from the
#                 earliest record it returns onwards are replaced; older rows are kept.
# 'incremental' = requery the trailing LOOKBACK_DAYS (for late arrivals) and merge by stable Id.
MODE          = 'incremental'
BACKFILL_DAYS = 30                                    # history to pull when MODE='backfill' (up to 180)
LOOKBACK_DAYS = 7                                     # trailing requery window + empty-table fallback
OUTPUT_TABLE  = 'dbo.Copilot_Interactions_Parsed'     # Delta table consumed by the PBIT (keep name/schema stable)

# --- Audit source --------------------------------------------------------------------
# Where the Copilot audit records come from. Every source writes the same rows to OUTPUT_TABLE, keyed by
# the audit record Id, so switching or mixing sources never duplicates a row.
# 'feed'     = Office 365 Management Activity API (Audit.General, the last 7 days). The default.
# 'auto'     = Microsoft Sentinel's CopilotActivity table for the hours it holds, otherwise the Office 365
#              Management Activity API for the last 7 days.
# 'graph'    = Microsoft Graph audit search for every window (any history; slow on a large tenant).
# 'sentinel' = Sentinel only.
# Except with 'graph', older windows are left out, so a first load covers 7 days; set GRAPH_FOR_HISTORY = True
# for one run to load older history from Graph audit search (needs AuditLogsQuery.Read.All).
AUDIT_SOURCE          = 'feed'
SENTINEL_QUERY_VIA    = 'hunting'           # 'hunting' (Graph advanced hunting, ThreatHunting.Read.All) or 'loganalytics'
SENTINEL_WORKSPACE_ID = ''                  # Log Analytics workspace ID: required for 'loganalytics' (app needs Log Analytics Reader); for 'hunting', blank = the primary workspace
SENTINEL_TABLE        = 'CopilotActivity'   # table the Microsoft Copilot data connector fills
GRAPH_FOR_HISTORY     = False               # True = windows no other source holds are read from Graph (slow; for a one-time history load)
FEED_HOST             = 'manage.office.com' # GCC: manage-gcc.office.com; GCC High: manage.office365.us; DoD: manage.protection.apps.mil
FEED_PUBLISHER_ID     = ''                  # optional: your tenant ID, for a dedicated Management API quota

# --- Backfill scale tuning ---------------------------------------------------------
CHUNK_HOURS             = 8     # window size; backfill = many windows. Use 2 on a very large tenant: 8h can fail in the audit service.
MAX_CONCURRENT_QUERIES  = 5     # Graph allows 50 audit searches queued or running per tenant and 200 new ones a day; 5 leaves room for other admins.
MAX_WAIT_MIN_PER_QUERY  = 240   # ceiling PER window (not the whole run). Resumable across reruns.
POLL_INTERVAL_SEC       = 30    # poll cadence per window
STAGING_DIR             = 'Files/_audit_staging'  # per-window JSONL + manifest; active runs only read their own windows.

# --- Failed-window recovery ----------------------------------------------------------
# A window whose query ends 'failed' or 'cancelled' (or times out) is sent again as a NEW query after a
# backoff. If it keeps failing it is split in half (8h -> 4h -> 2h -> 1h) and the halves are queried.
# Every window is attempted; the run only fails (without writing) if a window fails at MIN_CHUNK_HOURS.
WINDOW_RETRIES          = 3     # extra attempts per window before it is split
MIN_CHUNK_HOURS         = 1     # smallest size a failing window is split down to
SPLIT_ON_TIMEOUT        = True  # a window that hits MAX_WAIT_MIN_PER_QUERY splits at once instead of re-waiting
RETRY_BASE_SEC          = 60    # backoff before a retry: ~60s, 120s, 240s (with jitter) ...
RETRY_MAX_SEC           = 900   # ... never longer than this
MIN_CONCURRENT_QUERIES  = 1     # failures and HTTP 429s lower concurrency for the rest of the run, not below this
ALLOW_UNRECOVERED_GAPS  = False # incremental refuses to run while an older window failed and was never recovered

# --- Long runs (large tenants) -------------------------------------------------------
# With a budget, the run stops starting windows once it is used up and fails before anything is written,
# so the pipeline's retry carries on: finished windows are reused and queries still running in the audit
# service are picked up again rather than sent afresh. Keep it below the activity timeout, leaving time
# to read, parse and write what was fetched.
TIME_BUDGET_MIN         = 0     # minutes for fetching windows; 0 = no budget
RESUME_QUERY_HOURS      = 12    # a query an earlier attempt left running is re-polled if it is this recent
TRAILING_REUSE_HOURS    = 6     # a trailing window fetched this recently is reused, not queried again
# The audit service can answer a search with no records and add them later (for example during a service
# incident). A window still empty when it leaves the LOOKBACK_DAYS is queried once more if it ended in
# the last EMPTY_RECHECK_DAYS, oldest first and at most EMPTY_RECHECK_MAX a run (each one is an audit search).
EMPTY_RECHECK_DAYS      = 30    # 0 = never query an empty window again
EMPTY_RECHECK_MAX       = 24    # empty windows queried once more per run
# Graph throttles new audit searches per tenant (at least 200 a rolling 24 hours, shared with other admins).
# When it refuses one, the run stops starting windows, keeps its manifest and query ids, and fails before
# writing; the next run (the pipeline retry, or the next day's) carries on where this one stopped.
QUOTA_RETRY_AFTER_SEC   = 300   # a 429 asking to wait longer than this is the daily limit: pause, don't sleep

# --- Production: load secret from Key Vault instead of hardcoding -------------------
# from notebookutils.credentials import getSecret
# CLIENT_SECRET = getSecret('https://<your-vault>.vault.azure.net/', 'CopilotAuditAppSecret')


## 2. Authenticate to Microsoft Graph

Service principal (client secret) flow — same auth pattern as the PowerShell scripts.


In [ ]:
import base64, json, requests, time
from requests.adapters import HTTPAdapter
try:
    from urllib3.util.retry import Retry
except Exception:
    from requests.packages.urllib3.util.retry import Retry

# Resilient HTTP session. The Microsoft Graph security/auditLog endpoints intermittently return
# 502/503/504 (gateway timeouts) and 429 (throttling), especially during the poll + records-fetch
# phases on busy tenants. A shared Session with urllib3 Retry transparently retries those with
# exponential backoff + Retry-After, so a transient blip no longer aborts the whole run.
THROTTLE_LISTENERS = []  # step 5 registers its query limiter here; called on every HTTP 429.
# Step 5 registers the run's time-budget check here. Every retry wait below calls it, so a throttled
# or failing audit service can't hold the notebook past TIME_BUDGET_MIN (and the pipeline timeout).
BUDGET_CHECKS = []
QUOTA_RETRY_AFTER_SEC = int(globals().get('QUOTA_RETRY_AFTER_SEC', 300))


class QueryQuotaReached(RuntimeError):
    """Graph refused new audit searches for the tenant (HTTP 429): the window is left for the next run."""


def _wait(seconds):
    """Sleep for a retry, but stop as soon as a budget check raises."""
    end = time.monotonic() + max(0.0, float(seconds or 0))
    while True:
        for check in list(BUDGET_CHECKS):
            check()
        left = end - time.monotonic()
        if left <= 0:
            return
        time.sleep(min(left, 5))


def _notify_throttle():
    for listener in list(THROTTLE_LISTENERS):
        try:
            listener()
        except Exception:
            pass


class _ThrottleAwareRetry(Retry):
    # urllib3 retries 429s before _request sees them; report each one so concurrency can back off.
    def increment(self, method=None, url=None, response=None, *args, **kwargs):
        if response is not None and getattr(response, 'status', None) == 429:
            _notify_throttle()
        return super().increment(method, url, response, *args, **kwargs)

    def sleep(self, response=None):
        # Same wait urllib3 would use (Retry-After, else backoff), but cut short by the time budget.
        # A 429 asking for longer than QUOTA_RETRY_AFTER_SEC is the tenant's daily search limit: stop.
        wait = self.get_retry_after(response) if self.respect_retry_after_header and response is not None else None
        if wait is not None and wait > QUOTA_RETRY_AFTER_SEC and getattr(response, 'status', None) == 429:
            raise QueryQuotaReached(f'Graph asked to wait {wait:.0f}s before the next audit request (HTTP 429)')
        _wait(self.get_backoff_time() if wait is None else wait)


def _build_session() -> requests.Session:
    s = requests.Session()
    retry = _ThrottleAwareRetry(
        total=8,                       # up to 8 attempts per request
        connect=5, read=5, status=8,
        backoff_factor=2,              # 0,2,4,8,16,32... seconds (capped by urllib3)
        status_forcelist=(429, 500, 502, 503, 504),
        allowed_methods=frozenset(['GET', 'POST']),
        respect_retry_after_header=True,
        raise_on_status=False,
    )
    adapter = HTTPAdapter(max_retries=retry, pool_connections=8, pool_maxsize=8)
    s.mount('https://', adapter)
    s.mount('http://', adapter)
    return s

SESSION = _build_session()

# Belt-and-braces wrapper: urllib3 Retry handles most cases, but we add an outer retry loop so a
# request that *still* surfaces a transient status (or a connection error) after the adapter gives
# up is retried a few more times rather than killing the notebook.
_TRANSIENT = {429, 500, 502, 503, 504}

def _request(method: str, url: str, *, attempts: int = 5, retry_statuses=_TRANSIENT, **kwargs):
    kwargs.setdefault('timeout', 60)
    last = None
    for i in range(1, attempts + 1):
        try:
            r = SESSION.request(method, url, **kwargs)
        except requests.exceptions.RequestException as e:
            last = e
            if i == attempts:
                raise
            wait = min(60, 2 ** i)
            print(f'    transient network error ({type(e).__name__}); retry {i}/{attempts} in {wait}s')
            _wait(wait)
            continue
        if r.status_code == 429:
            _notify_throttle()
        if r.status_code in retry_statuses and i < attempts:
            wait = min(60, 2 ** i)
            print(f'    transient HTTP {r.status_code} on {method} {url.split("?")[0].rsplit("/",1)[-1]}; '
                  f'retry {i}/{attempts} in {wait}s')
            _wait(wait)
            continue
        return r
    return r  # pragma: no cover

# Auto-refreshing app-only Graph token. Long (multi-hour) chunked runs would otherwise
# fail when the 1-hour token expires mid-pull, so we transparently refresh every ~50 min.
def _get_graph_token() -> str:
    url  = f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token'
    body = {
        'client_id':     CLIENT_ID,
        'scope':         'https://graph.microsoft.com/.default',
        'client_secret': CLIENT_SECRET,
        'grant_type':    'client_credentials',
    }
    r = _request('POST', url, data=body, timeout=30)
    r.raise_for_status()
    return r.json()['access_token']

_token = {'value': None, 'ts': 0.0}

def get_headers() -> dict:
    if _token['value'] is None or (time.time() - _token['ts']) > 50 * 60:
        _token['value'] = _get_graph_token()
        _token['ts'] = time.time()
    return {'Authorization': f'Bearer {_token["value"]}', 'Content-Type': 'application/json'}

AUDIT_ROLE = 'AuditLogsQuery.Read.All'


def token_claims(token):
    """The claims of a JWT access token (not verified), or {} when it can't be read."""
    try:
        part = str(token).split('.')[1]
        claims = json.loads(base64.urlsafe_b64decode(part + '=' * (-len(part) % 4)).decode('utf-8'))
    except Exception:
        return {}
    return claims if isinstance(claims, dict) else {}


def audit_token_problems(token, tenant_id=None):
    """Why a Graph token can't read Copilot audit records: another tenant, or no AuditLogsQuery.Read.All.

    The per-service roles (AuditLogsQuery-Exchange.Read.All and the like) cover only their own
    service's records, not Copilot interactions. A token that can't be read gives no problems.
    """
    claims = token_claims(token)
    if not claims:
        return []
    problems = []
    tid = str(claims.get('tid') or '').strip().lower()
    want = str(tenant_id or '').strip().lower()
    if tid and len(want) == 36 and want.count('-') == 4 and tid != want:
        problems.append(f'the token is for tenant {tid}, not TENANT_ID {want}')
    roles = [r for r in (claims.get('roles') or []) if isinstance(r, str)]
    if AUDIT_ROLE not in roles:
        scoped = sorted(r for r in roles if r.startswith('AuditLogsQuery-'))
        if scoped:
            problems.append(f"the token has {', '.join(scoped)} but not {AUDIT_ROLE}; those roles cover only "
                            f'their own service, not Copilot interactions')
        else:
            problems.append(f'the token has no {AUDIT_ROLE} application permission')
    return problems


_scoped_tokens = {}


def get_scoped_headers(resource: str) -> dict:
    """App-only token headers for another API (the Management Activity API, Log Analytics), refreshed every ~50 min."""
    cached = _scoped_tokens.get(resource)
    if cached is None or (time.time() - cached[1]) > 50 * 60:
        r = _request('POST', f'https://login.microsoftonline.com/{TENANT_ID}/oauth2/v2.0/token', timeout=30, data={
            'client_id': CLIENT_ID, 'scope': f'{resource.rstrip("/")}/.default',
            'client_secret': CLIENT_SECRET, 'grant_type': 'client_credentials',
        })
        r.raise_for_status()
        cached = _scoped_tokens[resource] = (r.json()['access_token'], time.time())
    return {'Authorization': f'Bearer {cached[0]}', 'Content-Type': 'application/json'}


AUDIT_SOURCE = str(globals().get('AUDIT_SOURCE', 'graph') or 'graph').strip().lower()
if AUDIT_SOURCE not in ('graph', 'auto', 'sentinel', 'feed'):
    raise ValueError(f"Unsupported AUDIT_SOURCE={AUDIT_SOURCE!r}; expected 'graph', 'auto', 'sentinel' or 'feed'.")
GRAPH_FOR_HISTORY = bool(globals().get('GRAPH_FOR_HISTORY', False))

get_headers()
print('\u2713 Graph token acquired (auto-refreshing, retry-resilient session).')
# A token for the wrong tenant, or with only a per-service audit role, gets searches that succeed
# with no records. Stop here instead, with the fix. With another source, Graph only reads history,
# so a missing role is a warning: a window that needs Graph fails with the same fix.
_problems = audit_token_problems(_token['value'], TENANT_ID)
_role_problems = [p for p in _problems if not p.startswith('the token is for tenant')]
if _role_problems and AUDIT_SOURCE != 'graph':
    if AUDIT_SOURCE == 'auto' and GRAPH_FOR_HISTORY:
        print(f"Note: Graph can't read audit history ({'; '.join(_role_problems)}). Windows only Graph can read will fail; "
              f'add {AUDIT_ROLE} to the app, or set GRAPH_FOR_HISTORY = False to leave them out.')
    _problems = [p for p in _problems if p not in _role_problems]
if _problems:
    raise RuntimeError(
        'The Graph token cannot read Copilot audit records: ' + '; '.join(_problems) + '. Fix: in Entra -> '
        f'App registrations -> your app -> API permissions, add the Microsoft Graph application permission '
        f'{AUDIT_ROLE} and grant admin consent, then rerun this cell (a new permission can take a few '
        f'minutes to reach new tokens). TENANT_ID must be the tenant your Copilot users are in.'
    )
if token_claims(_token['value']) and not _role_problems:
    print(f"\u2713 Token is for tenant {token_claims(_token['value']).get('tid')} and has {AUDIT_ROLE}.")
print(f'Audit source: {AUDIT_SOURCE}')


## 3. Build query windows + the `create_query` helper

The lookback is split into smaller time windows (`CHUNK_HOURS`). Each window is its own async
audit query, which completes far faster than one giant query — the main cause of timeouts on
large tenants — and lets the run checkpoint window-by-window.

In [ ]:
from datetime import datetime, timezone, timedelta
import hashlib, json, random, threading, time

WINDOW_KEY_VERSION = 2
KEY_COLUMNS = ['Id', 'Source_RecordKey', 'Source_MessageKey', 'Source_ResourceKey']
_RESOURCE_KEY_FIELDS = ('Type', 'Action', 'SiteUrl', 'SensitivityLabelId')
_SYNTHETIC_AUDIT_FIELDS = frozenset({'_StableKey', '_StableOrdinal'})


def _as_utc_datetime(value):
    if value in (None, ''):
        return None
    if isinstance(value, datetime):
        dt = value
    else:
        text = str(value).strip()
        if not text:
            return None
        if text.endswith('Z'):
            text = text[:-1] + '+00:00'
        dt = datetime.fromisoformat(text)
    if dt.tzinfo is None:
        return dt.replace(tzinfo=timezone.utc)
    return dt.astimezone(timezone.utc)


def _norm_text(value) -> str:
    if value is None:
        return ''
    return ' '.join(str(value).strip().split())


def _coerce_json_value(value):
    if value is None:
        return None
    if isinstance(value, str):
        text = value.strip()
        if not text:
            return None
        try:
            return json.loads(text)
        except json.JSONDecodeError:
            return text
    return value


def _has_meaningful_value(value) -> bool:
    if isinstance(value, dict):
        return any(_has_meaningful_value(v) for k, v in value.items() if k not in _SYNTHETIC_AUDIT_FIELDS)
    if isinstance(value, list):
        return any(_has_meaningful_value(v) for v in value)
    if value is None:
        return False
    if isinstance(value, str):
        return _norm_text(value) != ''
    return True


def _canonical_json_text(value) -> str:
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(',', ':'))


def _stable_message_base_key(message) -> str:
    message = dict(message or {})
    msg_id = _norm_text(message.get('Id'))
    if msg_id:
        return f'mid:{msg_id}'
    payload = {k: v for k, v in message.items() if k not in _SYNTHETIC_AUDIT_FIELDS and k != 'Id'}
    return 'msgf:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def _stable_resource_base_key(resource) -> str:
    resource = dict(resource or {})
    payload = {k: v for k, v in resource.items() if k not in _SYNTHETIC_AUDIT_FIELDS}
    if not _has_meaningful_value(payload):
        return 'resource:none'
    return 'resource:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def _stable_sequence(items, base_key_builder, path) -> list:
    prepared = []
    for item in items or []:
        canonical_item = _canonicalize_json(item, path)
        if not isinstance(canonical_item, dict):
            canonical_item = {'Value': canonical_item}
        canonical_text = _canonical_json_text(canonical_item)
        base_key = base_key_builder(canonical_item)
        prepared.append((base_key, canonical_text, canonical_item))
    prepared.sort(key=lambda entry: (entry[0], entry[1]))
    totals = {}
    for base_key, _canonical_text, _item in prepared:
        totals[base_key] = totals.get(base_key, 0) + 1
    seen = {}
    stable = []
    for ordinal, (base_key, _canonical_text, canonical_item) in enumerate(prepared):
        seen[base_key] = seen.get(base_key, 0) + 1
        stable_key = base_key if totals[base_key] == 1 else f'{base_key}#{seen[base_key]}'
        enriched = dict(canonical_item)
        enriched['_StableKey'] = stable_key
        enriched['_StableOrdinal'] = ordinal
        stable.append(enriched)
    return stable


def _canonicalize_json(value, path=()):
    value = _coerce_json_value(value) if not path else value
    if isinstance(value, dict):
        return {
            str(key): _canonicalize_json(val, path + (str(key),))
            for key, val in sorted(value.items(), key=lambda item: str(item[0]))
            if key not in _SYNTHETIC_AUDIT_FIELDS
        }
    if isinstance(value, list):
        if path == ('CopilotEventData', 'Messages'):
            return _stable_sequence(value, _stable_message_base_key, path)
        if path == ('CopilotEventData', 'AccessedResources'):
            return _stable_sequence(value, _stable_resource_base_key, path)
        return [_canonicalize_json(item, path) for item in value]
    return value


def canonicalize_audit_payload(audit_data):
    value = _coerce_json_value(audit_data)
    if value is None:
        return None
    return _canonicalize_json(value)


def determine_incremental_start(end_dt, high_water_mark, lookback_days: int):
    lookback_days = max(int(lookback_days), 0)
    lookback_floor = end_dt - timedelta(days=lookback_days)
    hw = _as_utc_datetime(high_water_mark)
    if hw is None:
        return lookback_floor
    return min(hw, lookback_floor)


def validate_incremental_target_columns(columns, required_columns, table_name: str):
    cols = set(columns or [])
    missing = [c for c in required_columns if c not in cols]
    if missing:
        raise RuntimeError(
            f"Legacy table {table_name} is missing stable key columns {missing}. "
            "Run MODE='backfill' once before incremental resumes: a backfill replaces a table "
            "without these columns whole, which upgrades it."
        )


def build_source_record_key(
    record_id,
    creation_date=None,
    operation=None,
    audit_data=None,
    record_type=None,
    associated_admin_units=None,
    associated_admin_units_names=None,
) -> str:
    rid = _norm_text(record_id)
    if rid:
        return f"rid:{rid}"
    payload = {
        'CreationDate': _norm_text(creation_date),
        'Operation': _norm_text(operation),
        'RecordType': _norm_text(record_type),
        'AuditData': canonicalize_audit_payload(audit_data),
        'AssociatedAdminUnits': _canonicalize_json(associated_admin_units or []),
        'AssociatedAdminUnitsNames': _canonicalize_json(associated_admin_units_names or []),
    }
    return 'synthetic:' + hashlib.sha256(_canonical_json_text(payload).encode('utf-8')).hexdigest()


def canonicalize_audit_record(record) -> dict:
    record = record or {}
    audit_payload = canonicalize_audit_payload(record.get('auditData'))
    return {
        'RecordId': record.get('id'),
        'CreationDate': record.get('createdDateTime'),
        'RecordType': record.get('auditLogRecordType'),
        'Operation': record.get('operation'),
        'AuditData': _canonical_json_text(audit_payload) if audit_payload is not None else None,
        'SourceRecordKey': build_source_record_key(
            record.get('id'),
            record.get('createdDateTime'),
            record.get('operation'),
            audit_payload,
            record_type=record.get('auditLogRecordType'),
            associated_admin_units=record.get('associatedAdminUnits', []),
            associated_admin_units_names=record.get('associatedAdminUnitsNames', []),
        ),
        'AssociatedAdminUnits': _canonical_json_text(_canonicalize_json(record.get('associatedAdminUnits', []))),
        'AssociatedAdminUnitsNames': _canonical_json_text(_canonicalize_json(record.get('associatedAdminUnitsNames', []))),
    }


def stable_window_key(win_start, win_end) -> str:
    ws = _as_utc_datetime(win_start)
    we = _as_utc_datetime(win_end)
    return f"v{WINDOW_KEY_VERSION}_{ws:%Y%m%d%H%M}_{we:%Y%m%d%H%M}"


# --- Failed-window recovery: kept verbatim with core/valuelens_core/audit.py -------------
# A split window keeps its own stable_window_key; its halves get theirs, so manifests written before
# splitting existed stay valid and every part is reusable on a rerun.
def split_window(win_start, win_end, min_hours):
    """Halves of a window for another try at a smaller size; [] when a half would be under min_hours."""
    ws = _as_utc_datetime(win_start)
    we = _as_utc_datetime(win_end)
    half_minutes = int((we - ws).total_seconds() // 60) // 2
    if half_minutes <= 0 or half_minutes < float(min_hours) * 60:
        return []
    mid = win_start + timedelta(minutes=half_minutes)
    return [(win_start, mid), (mid, win_end)]


def expand_split_windows(manifest, win_start, win_end, min_hours, include_parents=False):
    """The windows to query for one grid window: its halves (recursively) if an earlier run split it."""
    entry = (manifest or {}).get(stable_window_key(win_start, win_end))
    halves = []
    if isinstance(entry, dict) and entry.get('status') == 'split':
        halves = split_window(win_start, win_end, min_hours)
    if not halves:
        return [(win_start, win_end)]
    out = [(win_start, win_end)] if include_parents else []
    for ws, we in halves:
        out.extend(expand_split_windows(manifest, ws, we, min_hours, include_parents))
    return out


def retry_delay(attempt, base_seconds, max_seconds, rand=random.random):
    """Exponential backoff with jitter: base, 2x base, 4x base ... capped at max_seconds, times 50-100%."""
    raw = min(float(max_seconds), float(base_seconds) * (2 ** max(int(attempt) - 1, 0)))
    return max(0.0, raw * (0.5 + 0.5 * rand()))


class AdaptiveLimiter:
    """Caps concurrent audit queries. Failures and throttling lower the cap for the rest of the run.

    A burst of failures with one cause (several windows failing in the same second) counts once:
    the cap moves at most once per cooldown_seconds.

    check, if given, is called while a caller waits for a free slot; raising from it ends the wait
    (the run's time budget uses this, so a throttled run can't wait past it).
    """

    def __init__(self, limit, minimum=1, cooldown_seconds=60, clock=time.monotonic, check=None):
        self.minimum = max(1, int(minimum))
        self.limit = max(self.minimum, int(limit))
        self.active = 0
        self.history = []
        self._cooldown = float(cooldown_seconds)
        self._clock = clock
        self._last_shrink = None
        self._check = check
        self._cond = threading.Condition()

    def __enter__(self):
        with self._cond:
            while self.active >= self.limit:
                if self._check is not None:
                    self._check()
                self._cond.wait(None if self._check is None else 5)
            self.active += 1
        return self

    def __exit__(self, *exc):
        with self._cond:
            self.active -= 1
            self._cond.notify_all()
        return False

    def shrink(self, reason, halve=False):
        with self._cond:
            now = self._clock()
            if self._last_shrink is not None and now - self._last_shrink < self._cooldown:
                return False
            new = max(self.minimum, self.limit // 2 if halve else self.limit - 1)
            if new >= self.limit:
                return False
            self.history.append((self.limit, new, reason))
            self.limit, self._last_shrink = new, now
            return True


def _manifest_range(entry):
    if not isinstance(entry, dict):
        return None
    try:
        ws = _as_utc_datetime(entry.get('window_start'))
        we = _as_utc_datetime(entry.get('window_end'))
    except (TypeError, ValueError):
        return None
    return (ws, we) if ws and we and ws < we else None


RECHECK_PENDING = 'recheck_pending'


def uncovered_failed_ranges(manifest, not_before, before):
    """Time ranges in [not_before, before) of unfinished windows that no succeeded window covers.

    A window flagged RECHECK_PENDING counts as covered: every run plans it again until it is written.
    """
    lo = _as_utc_datetime(not_before)
    hi = _as_utc_datetime(before)
    entries = [e for e in (manifest or {}).values() if isinstance(e, dict)]
    covered = sorted(r for r in (_manifest_range(e) for e in entries
                                 if e.get('status') == 'succeeded' or e.get(RECHECK_PENDING)) if r)
    gaps = []
    for entry in entries:
        if entry.get('status') in ('succeeded', 'split'):
            continue
        rng = _manifest_range(entry)
        if not rng:
            continue
        pieces = [(max(rng[0], lo), min(rng[1], hi))]
        for cs, ce in covered:
            pieces = [p for s, e in pieces for p in ((s, min(e, cs)), (max(s, ce), e)) if p[0] < p[1]]
        gaps.extend(pieces)
    merged = []
    for s, e in sorted(gaps):
        if merged and s <= merged[-1][1]:
            merged[-1] = (merged[-1][0], max(merged[-1][1], e))
        else:
            merged.append((s, e))
    return merged


def empty_recheck_windows(manifest, now, trailing_cutoff, recheck_days, settle_days):
    """Succeeded windows that came back empty and are due one more query, oldest first.

    The audit service can answer a search with no records and fill the hours in later (a service
    incident, a slow tenant). The trailing look-back re-queries recent windows; a window still empty
    when it leaves the look-back, and last fetched within `settle_days` of its end, is due once more if
    it ended in the last `recheck_days`. A later fetch of the same hours supersedes it.
    """
    if not recheck_days or recheck_days <= 0:
        return []
    oldest = _as_utc_datetime(now) - timedelta(days=recheck_days)
    cutoff = _as_utc_datetime(trailing_cutoff)
    settle = timedelta(days=max(int(settle_days or 0), 1))
    done = []
    for entry in (manifest or {}).values():
        if not isinstance(entry, dict) or entry.get('status') != 'succeeded':
            continue
        rng = _manifest_range(entry)
        completed = _as_utc_datetime(entry.get('completed_at'))
        if rng and completed:
            done.append((rng[0], rng[1], completed, entry))
    due = set()
    for ws, we, completed, entry in done:
        if int(entry.get('rows', 0) or 0) or int(entry.get('pages', 0) or 0) or entry.get('files'):
            continue
        if not (oldest < we <= cutoff) or completed >= we + settle:
            continue
        pieces = [(ws, we)]
        for cs, ce, later, _ in done:
            if later > completed:
                pieces = [p for s, e in pieces for p in ((s, min(e, cs)), (max(s, ce), e)) if p[0] < p[1]]
        if pieces:
            due.add((ws, we))
    return sorted(due)


def pending_recheck_windows(manifest):
    """Windows flagged RECHECK_PENDING, oldest first: re-queried, but their rows not yet written.

    A run that fails after a recheck keeps the flag, so the next run plans the window again (reusing what
    was fetched) and writes it; the flag is cleared once the run's rows are written.
    """
    return sorted({rng for rng in (_manifest_range(e) for e in (manifest or {}).values()
                                   if isinstance(e, dict) and e.get(RECHECK_PENDING)) if rng})


if MODE not in ('backfill', 'incremental'):
    raise ValueError(f"Unsupported MODE={MODE!r}; expected 'backfill' or 'incremental'.")

end_date = datetime.now(timezone.utc)
if MODE == 'backfill':
    start_date = end_date - timedelta(days=BACKFILL_DAYS)
    WRITE_MODE = 'replace'
else:
    WRITE_MODE = 'merge'
    _hw = None
    if spark.catalog.tableExists(OUTPUT_TABLE):
        from pyspark.sql import functions as F
        _existing = spark.table(OUTPUT_TABLE)
        validate_incremental_target_columns(_existing.columns, KEY_COLUMNS, OUTPUT_TABLE)
        _hw = _existing.agg(F.max('CreationDate').alias('m')).collect()[0]['m']
    start_date = determine_incremental_start(end_date, _hw, LOOKBACK_DAYS)
    if _hw:
        print(
            f'incremental: high-water mark = {_as_utc_datetime(_hw):%Y-%m-%d %H:%M}; '
            f'requerying trailing {LOOKBACK_DAYS}d from {start_date:%Y-%m-%d %H:%M}'
        )
    else:
        print(f'incremental: no prior data, defaulting to last {LOOKBACK_DAYS}d')

_anchor_hour = start_date.hour - (start_date.hour % CHUNK_HOURS)
_cur = start_date.replace(hour=_anchor_hour, minute=0, second=0, microsecond=0)
windows = []
while _cur < end_date:
    _nxt = min(_cur + timedelta(hours=CHUNK_HOURS), end_date)
    windows.append((_cur, _nxt))
    _cur = _nxt
print(f'MODE={MODE}: {start_date:%Y-%m-%d} -> {end_date:%Y-%m-%d}  =>  {len(windows)} window(s) of {CHUNK_HOURS}h, WRITE_MODE={WRITE_MODE}')


def create_query(win_start, win_end) -> str:
    body = {
        'displayName':         f'Copilot Interactions {win_start:%Y%m%d%H%M}-{win_end:%Y%m%d%H%M}',
        'filterStartDateTime': win_start.isoformat(),
        'filterEndDateTime':   win_end.isoformat(),
        'recordTypeFilters':   ['copilotInteraction'],
        'operationFilters':    [],
    }
    # urllib3 has already retried a 429 eight times, honouring Retry-After; another 429 means the
    # tenant's audit search limit, so the run pauses instead of sending more.
    r = _request('POST', 'https://graph.microsoft.com/beta/security/auditLog/queries',
                 json=body, headers=get_headers(), timeout=30, retry_statuses=_TRANSIENT - {429})
    if r.status_code == 429:
        raise QueryQuotaReached('Graph refused a new audit search for the tenant (HTTP 429)')
    r.raise_for_status()
    return r.json()['id']


## 4. Resumable wait helper

Polls an async query until it succeeds, with a **generous, configurable** ceiling
(`MAX_WAIT_MIN_PER_QUERY`) instead of a hard 30-minute cut-off. The token auto-refreshes while
waiting, so multi-hour jobs don't fail on expiry.

A query the service ends as `failed` or `cancelled` raises `AuditQueryFailed`. Step 5 retries it with a
fresh query, and splits the window if it keeps failing.

In [ ]:
import time


class AuditQueryFailed(RuntimeError):
    """The audit service ended a query as failed or cancelled; a new query for the window may succeed."""


class BudgetReached(RuntimeError):
    """TIME_BUDGET_MIN ran out. The window is left for the retry, which re-polls its query."""


class RecordLimitExceeded(RuntimeError):
    """The query hit the audit service's record cap (isRecordCountLimitExceeded), so its records are
    incomplete. The window is split and its halves queried."""


def budget_exhausted(deadline, now=None) -> bool:
    return deadline is not None and (time.monotonic() if now is None else now) >= deadline


RUN_DEADLINE = time.monotonic() + TIME_BUDGET_MIN * 60 if TIME_BUDGET_MIN and TIME_BUDGET_MIN > 0 else None


def check_budget():
    """Raise BudgetReached once TIME_BUDGET_MIN has run out. Retry waits and the query limiter call this."""
    if budget_exhausted(RUN_DEADLINE):
        raise BudgetReached('Time budget reached while waiting on the audit service')


def _read_query_status(query_id: str):
    r = _request(
        'GET',
        f'https://graph.microsoft.com/beta/security/auditLog/queries/{query_id}',
        headers=get_headers(),
        timeout=30,
    )
    r.raise_for_status()
    payload = r.json()
    if not isinstance(payload, dict):
        raise RuntimeError(
            f'Query {query_id} returned malformed status payload type: {type(payload).__name__}'
        )
    status = payload.get('status')
    if not isinstance(status, str) or not status.strip():
        raise RuntimeError(f'Query {query_id} returned missing/invalid status: {status!r}')
    return payload, status.strip().lower()


def wait_for_query(query_id: str, max_wait_min: int = MAX_WAIT_MIN_PER_QUERY):
    started = time.time()
    deadline = started + max_wait_min * 60
    polls = 0
    while time.time() < deadline:
        if budget_exhausted(RUN_DEADLINE):
            raise BudgetReached(f'Time budget reached while query {query_id} was still running')
        polls += 1
        payload, status = _read_query_status(query_id)
        if status == 'succeeded':
            return payload
        if status in ('failed', 'cancelled'):
            raise AuditQueryFailed(f'Query {query_id} ended with status: {status}')
        if status not in ('notstarted', 'running'):
            raise RuntimeError(f'Query {query_id} returned unexpected status: {status}')
        if polls % 4 == 0:
            mins = int((time.time() - started) // 60)
            print(f'    … still {status} ({mins}m elapsed, {polls} polls)')
        time.sleep(POLL_INTERVAL_SEC)
    raise TimeoutError(
        f'Query {query_id} did not succeed within {max_wait_min} min. '
        f'Increase MAX_WAIT_MIN_PER_QUERY or reduce CHUNK_HOURS.'
    )


## 5. Fetch records → stream to Lakehouse Files (chunked, bounded memory)

Each page of results is written straight to Lakehouse **Files** as JSON-lines, so the full
result set is **never** held in driver memory — the other main cause of OOM/timeouts on large
tenants.

**Failed windows recover on their own.** On large tenants a query sometimes ends `failed` or `cancelled`,
often when several queries are submitted at once or someone else is running audit searches. Each failing
window is:

1. sent again as a **new** query, up to `WINDOW_RETRIES` times, with exponential backoff and jitter;
2. then **split in half** (8h → 4h → 2h → 1h, down to `MIN_CHUNK_HOURS`), and the halves are queried.
   The halves are recorded in the manifest, so a rerun reuses every half that already succeeded.

Each failure or HTTP 429 also lowers the number of concurrent queries for the rest of the run
(not below `MIN_CONCURRENT_QUERIES`). Every window is attempted before the run reports anything.
If a window still fails at `MIN_CHUNK_HOURS`, the cell stops with one message listing the failed windows
and **nothing is written**. Rerun the notebook (in `MODE='backfill'` if this was a backfill); succeeded
windows are reused, so only the failed ones are queried again. An incremental run refuses to start while
an older window is unrecovered, so switching modes can't silently leave a gap.

**Other sources** (`AUDIT_SOURCE` other than `'graph'`): the cell first finds the hours each source holds.
Sentinel windows are read with paged KQL queries (no audit search to wait for). The Management Activity API is
read once for all its windows: its content is listed by when it was published, so each record goes to the window
holding its time, and `_feed_state.json` in the staging folder remembers what was read, so the next run reads only
new content. If Sentinel can't be read under `'auto'`, it is skipped with a warning and the Management Activity API
reads its windows; if the Management Activity API can't be read, the run stops with the fix (unless
`GRAPH_FOR_HISTORY = True`, when Graph reads its windows). Windows no source holds are left out before any query.
The manifest records each window's `source`, which the Load Status notebook shows.

In [ ]:
import bisect, json, os, re, tempfile, threading, time
from urllib.parse import urlencode, urlparse
from concurrent.futures import FIRST_COMPLETED, ThreadPoolExecutor, wait as wait_futures

# Defaults for config cells copied from an older version of this notebook.
WINDOW_RETRIES = int(globals().get('WINDOW_RETRIES', 3))
MIN_CHUNK_HOURS = float(globals().get('MIN_CHUNK_HOURS', 1))
SPLIT_ON_TIMEOUT = bool(globals().get('SPLIT_ON_TIMEOUT', True))
RETRY_BASE_SEC = float(globals().get('RETRY_BASE_SEC', 60))
RETRY_MAX_SEC = float(globals().get('RETRY_MAX_SEC', 900))
MIN_CONCURRENT_QUERIES = int(globals().get('MIN_CONCURRENT_QUERIES', 1))
ALLOW_UNRECOVERED_GAPS = bool(globals().get('ALLOW_UNRECOVERED_GAPS', False))
# Set once Graph refuses a new audit search: the run then starts no more queries (running ones finish).
QUOTA_PAUSED = threading.Event()

# Resumable, bounded-concurrency backfill. Each window's records stream to a JSONL file and its
# status is recorded in a manifest; rerun skips older succeeded windows, but recent LOOKBACK_DAYS
# windows are intentionally re-queried for late arrivals. Active runs only read the current window set.
# A failing window is retried with a new query, then split into halves (status 'split', parts=[keys]);
# the halves have their own manifest entries and staging files.

def _normalize_stage_dir(path: str) -> str:
    if not isinstance(path, str) or not path.strip():
        raise ValueError(f'Stage directory must be a non-empty string; found {path!r}.')
    text = path.strip()
    if text.startswith('abfss://'):
        return text.rstrip('/')
    normalized = text.replace('\\', '/')
    mount_prefix = '/lakehouse/default/'
    if normalized.startswith(mount_prefix):
        normalized = normalized[len(mount_prefix):]
    if normalized == 'Files' or normalized.startswith('Files/'):
        return normalized.rstrip('/')
    return os.path.abspath(text)


def _is_remote_stage_dir(path: str) -> bool:
    return isinstance(path, str) and (path == 'Files' or path.startswith('Files/') or path.startswith('abfss://'))


def _stage_path_join(base_dir: str, name: str) -> str:
    if _is_remote_stage_dir(base_dir):
        return base_dir.rstrip('/') + '/' + name.lstrip('/')
    return os.path.join(base_dir, name)


def _stage_basename(path: str) -> str:
    if not isinstance(path, str) or not path.strip():
        raise ValueError(f'Stage path must be a non-empty string; found {path!r}.')
    return path.rstrip('/').replace('\\', '/').split('/')[-1]


def _file_uri(path: str) -> str:
    return 'file:' + os.path.abspath(path)


def _driver_temp_root(base_dir: str) -> str:
    configured = globals().get('LOCAL_STAGE_TMP_DIR')
    if isinstance(configured, str) and configured.strip():
        root = os.path.abspath(configured)
    elif _is_remote_stage_dir(base_dir):
        root = os.path.abspath(os.path.join(tempfile.gettempdir(), 'valuelens_audit_stage'))
    else:
        root = os.path.abspath(base_dir)
    os.makedirs(root, exist_ok=True)
    return root


def _new_local_temp_path(base_dir: str, stem: str, suffix: str) -> str:
    fd, temp_path = tempfile.mkstemp(
        prefix=f'.{stem}.{os.getpid()}.{threading.get_ident()}.',
        suffix=suffix,
        dir=_driver_temp_root(base_dir),
    )
    os.close(fd)
    return temp_path


def _get_notebook_fs():
    fs = globals().get('NOTEBOOK_FS')
    if fs is None:
        try:
            import notebookutils
        except ImportError as e:
            raise RuntimeError('Remote staging paths require notebookutils.fs in the Fabric runtime.') from e
        fs = getattr(notebookutils, 'fs', None)
    if fs is None:
        raise RuntimeError('Remote staging paths require notebookutils.fs.')
    missing = [name for name in ('ls', 'exists', 'rm', 'cp', 'mv', 'mkdirs') if not hasattr(fs, name)]
    if missing:
        raise RuntimeError(f'notebookutils.fs is missing required method(s): {missing}.')
    return fs


def _ensure_stage_dir(base_dir: str):
    if _is_remote_stage_dir(base_dir):
        fs = _get_notebook_fs()
        if not fs.exists(base_dir):
            fs.mkdirs(base_dir)
        return
    os.makedirs(base_dir, exist_ok=True)


def _list_stage_paths(base_dir: str) -> list:
    if _is_remote_stage_dir(base_dir):
        fs = _get_notebook_fs()
        if not fs.exists(base_dir):
            return []
        paths = []
        for entry in fs.ls(base_dir):
            path = getattr(entry, 'path', None)
            if path is None and isinstance(entry, dict):
                path = entry.get('path')
            if not isinstance(path, str) or not path.strip():
                raise RuntimeError(f'Stage listing entry for {base_dir} is missing a usable path: {entry!r}')
            paths.append(path.rstrip('/'))
        return sorted(set(paths))
    if not os.path.isdir(base_dir):
        return []
    return sorted(os.path.join(base_dir, name) for name in os.listdir(base_dir))


def _stage_exists(path: str) -> bool:
    if _is_remote_stage_dir(path):
        return _get_notebook_fs().exists(path)
    return os.path.exists(path)


def _stage_remove(path: str):
    if _is_remote_stage_dir(path):
        fs = _get_notebook_fs()
        if fs.exists(path):
            if fs.rm(path, False) is False:
                raise RuntimeError(f'Failed to remove staged file {path}.')
        return
    if os.path.exists(path):
        os.remove(path)


def _write_stage_text(base_dir: str, target_path: str, text: str):
    _ensure_stage_dir(base_dir)
    local_temp = _new_local_temp_path(base_dir, _stage_basename(target_path), '.tmp')
    try:
        with open(local_temp, 'w', encoding='utf-8') as f:
            f.write(text)
            f.flush()
            os.fsync(f.fileno())
        if _is_remote_stage_dir(base_dir):
            fs = _get_notebook_fs()
            _stage_remove(target_path)
            copied = fs.cp(_file_uri(local_temp), target_path)
            if copied is False:
                raise RuntimeError(f'Failed to copy local staging file to {target_path}.')
        else:
            os.replace(local_temp, target_path)
            local_temp = None
    finally:
        if local_temp and os.path.exists(local_temp):
            os.remove(local_temp)


def _write_stage_jsonl(base_dir: str, target_path: str, records):
    _ensure_stage_dir(base_dir)
    local_temp = _new_local_temp_path(base_dir, _stage_basename(target_path), '.jsonl')
    try:
        with open(local_temp, 'w', encoding='utf-8') as f:
            for record in records:
                f.write(json.dumps(_row(record)) + '\n')
            f.flush()
            os.fsync(f.fileno())
        if _is_remote_stage_dir(base_dir):
            fs = _get_notebook_fs()
            _stage_remove(target_path)
            copied = fs.cp(_file_uri(local_temp), target_path)
            if copied is False:
                raise RuntimeError(f'Failed to copy local staging page to {target_path}.')
        else:
            os.replace(local_temp, target_path)
            local_temp = None
    finally:
        if local_temp and os.path.exists(local_temp):
            os.remove(local_temp)


def _move_stage_file(base_dir: str, source_path: str, target_path: str, overwrite: bool = True):
    if _is_remote_stage_dir(base_dir):
        fs = _get_notebook_fs()
        # Fabric's third argument is create_path, not overwrite. Rename in place
        # without first deleting the valid destination checkpoint.
        moved = fs.mv(source_path, target_path, False, overwrite)
        if moved is False:
            raise RuntimeError(f'Failed to move staged file {source_path} to {target_path}.')
        return
    if not overwrite and os.path.exists(target_path):
        raise FileExistsError(target_path)
    os.replace(source_path, target_path)


def _read_stage_json(base_dir: str, source_path: str):
    if _is_remote_stage_dir(base_dir):
        local_temp = _new_local_temp_path(base_dir, _stage_basename(source_path), '.json')
        try:
            copied = _get_notebook_fs().cp(source_path, _file_uri(local_temp))
            if copied is False:
                raise RuntimeError(f'Failed to copy staged file {source_path} to local scratch.')
            with open(local_temp, encoding='utf-8') as f:
                return json.load(f)
        finally:
            if os.path.exists(local_temp):
                os.remove(local_temp)
    with open(source_path, encoding='utf-8') as f:
        return json.load(f)


STAGING_ABS = _normalize_stage_dir(STAGING_DIR)
MANIFEST = _stage_path_join(STAGING_ABS, '_manifest.json')
MANIFEST_LOCK = threading.RLock()
# Windows this run has written. Any other window another run saved meanwhile is kept, not overwritten.
MANIFEST_TOUCHED = set()
_ensure_stage_dir(STAGING_ABS)


def list_window_files(base_dir: str, key: str, include_partial: bool = False):
    prefix = f'win_{key}_'
    matches = []
    for path in _list_stage_paths(base_dir):
        name = _stage_basename(path)
        if not name.startswith(prefix):
            continue
        if name.endswith('.jsonl') or (include_partial and name.endswith('.jsonl.partial')):
            matches.append(path)
    return sorted(set(matches))


def purge_window_files(base_dir: str, key: str, include_partial: bool = True) -> int:
    removed = 0
    for path in list_window_files(base_dir, key, include_partial=include_partial):
        if _stage_exists(path):
            _stage_remove(path)
            removed += 1
    return removed


def list_active_stage_files(base_dir: str, active_keys) -> list:
    files = []
    for key in sorted(set(active_keys or [])):
        files.extend(list_window_files(base_dir, key, include_partial=False))
    return sorted(files)


def _allowed_manifest_statuses():
    return {'querying', 'waiting', 'draining', 'succeeded', 'failed', 'split'}


def _validate_manifest_payload(data):
    if not isinstance(data, dict):
        raise RuntimeError(
            f'Manifest {MANIFEST} must be a JSON object keyed by window id; found {type(data).__name__}.'
        )
    allowed = _allowed_manifest_statuses()
    for key, entry in data.items():
        if not isinstance(key, str) or not key.strip():
            raise RuntimeError(f'Manifest {MANIFEST} has an invalid window key: {key!r}.')
        if not isinstance(entry, dict):
            raise RuntimeError(
                f'Manifest {MANIFEST} entry {key!r} must be a JSON object; found {type(entry).__name__}.'
            )
        status = entry.get('status')
        if not isinstance(status, str) or status not in allowed:
            raise RuntimeError(
                f"Manifest {MANIFEST} entry {key!r} has invalid status {status!r}; "
                f'expected one of {sorted(allowed)}.'
            )
        if status == 'split':
            parts = entry.get('parts')
            if not isinstance(parts, list) or not parts or not all(isinstance(p, str) and p.strip() for p in parts):
                raise RuntimeError(f'Manifest {MANIFEST} entry {key!r} is split but has no valid parts list.')
    return data


def _load_manifest():
    if not _stage_exists(MANIFEST):
        return {}
    try:
        data = _read_stage_json(STAGING_ABS, MANIFEST)
    except json.JSONDecodeError as e:
        raise RuntimeError(f'Manifest {MANIFEST} is not valid JSON: {e}') from e
    except UnicodeDecodeError as e:
        raise RuntimeError(f'Manifest {MANIFEST} is not valid UTF-8 JSON: {e}') from e
    return _validate_manifest_payload(data)


def _manifest_temp_path():
    base = _stage_basename(MANIFEST)
    return _stage_path_join(STAGING_ABS, f'.{base}.{os.getpid()}.{threading.get_ident()}.tmp')


def _save_manifest(m):
    payload = _validate_manifest_payload(dict(m))
    _ensure_stage_dir(STAGING_ABS)
    temp_path = _manifest_temp_path()
    try:
        _write_stage_text(STAGING_ABS, temp_path, json.dumps(payload, indent=2, sort_keys=True))
        _move_stage_file(STAGING_ABS, temp_path, MANIFEST, overwrite=True)
    finally:
        if _stage_exists(temp_path):
            _stage_remove(temp_path)


def manifest_succeeded(entry) -> bool:
    return isinstance(entry, dict) and entry.get('status') == 'succeeded'


def should_refresh_window(win_end, manifest_entry, trailing_cutoff, fresh_after=None) -> bool:
    """Whether a window must be queried again. Trailing windows are, for late records, unless they
    were fetched after `fresh_after` (a retry soon after the run that fetched them)."""
    if not manifest_succeeded(manifest_entry):
        return True
    completed_at = _as_utc_datetime((manifest_entry or {}).get('completed_at'))
    if completed_at is None:
        return True
    if _as_utc_datetime(win_end) <= _as_utc_datetime(trailing_cutoff):
        return False
    return fresh_after is None or completed_at < _as_utc_datetime(fresh_after)


def trailing_fresh_after(now, reuse_hours):
    return now - timedelta(hours=reuse_hours) if reuse_hours and reuse_hours > 0 else None


def resumable_query_id(entry, now, max_hours):
    """The query an earlier attempt left running for this window, if it is recent enough to re-poll."""
    entry = entry or {}
    qid = entry.get('query_id')
    if entry.get('status') not in ('querying', 'waiting', 'draining') or not max_hours or max_hours <= 0:
        return None
    if not isinstance(qid, str) or not qid.strip():
        return None
    created = _as_utc_datetime(entry.get('query_created_at') or entry.get('refreshed_at'))
    if created is None or now - created > timedelta(hours=max_hours):
        return None
    return qid.strip()


def window_stage_files_reusable(base_dir: str, key: str, manifest_entry) -> bool:
    entry = manifest_entry or {}
    recorded = entry.get('files')
    if isinstance(recorded, list):
        if recorded:
            expected = [_stage_path_join(base_dir, _stage_basename(path)) for path in recorded]
            return all(_stage_exists(path) for path in expected)
        return int(entry.get('rows', 0) or 0) == 0 and int(entry.get('pages', 0) or 0) == 0
    files = list_window_files(base_dir, key, include_partial=False)
    if files:
        return True
    return int(entry.get('rows', 0) or 0) == 0 and int(entry.get('pages', 0) or 0) == 0


def is_window_reusable(base_dir: str, key: str, win_end, manifest_entry, trailing_cutoff, fresh_after=None) -> bool:
    return (not should_refresh_window(win_end, manifest_entry, trailing_cutoff, fresh_after)) and window_stage_files_reusable(
        base_dir, key, manifest_entry
    )


def _read_manifest_entry(key: str) -> dict:
    with MANIFEST_LOCK:
        return dict(manifest.get(key, {}))


def _manifest_entry_time(entry):
    stamps = [_as_utc_datetime(entry.get(k)) for k in ('completed_at', 'refreshed_at', 'paused_at', 'query_created_at')
              if isinstance(entry, dict) and entry.get(k)]
    return max(stamps) if stamps else datetime.min.replace(tzinfo=timezone.utc)


def _merge_other_runs(current: dict) -> dict:
    """Merge newer saved entries for windows this run hasn't written. Runs must not overlap."""
    saved = _load_manifest()
    for key, entry in saved.items():
        if key in MANIFEST_TOUCHED:
            continue
        mine = current.get(key)
        if mine is None or _manifest_entry_time(entry) > _manifest_entry_time(mine):
            current[key] = entry
    return current


def _mark_window(key: str, status: str, **extra):
    if not isinstance(key, str) or not key.strip():
        raise ValueError(f'Window key must be a non-empty string; found {key!r}.')
    if not isinstance(status, str) or status not in _allowed_manifest_statuses():
        raise ValueError(f'Unsupported manifest status {status!r}.')
    with MANIFEST_LOCK:
        # This run's own entries always come from memory: remote listings or mounted reads can lag a
        # successful publish, and reloading them could discard newer checkpoints from other workers.
        current = _merge_other_runs(dict(manifest))
        entry = dict(current.get(key, {}))
        entry.update(extra)
        entry['status'] = status
        current[key] = entry
        MANIFEST_TOUCHED.add(key)
        _save_manifest(current)
        manifest.clear()
        manifest.update(current)


def _row(r):
    return canonicalize_audit_record(r)


def _is_graph_records_url(url: str, qid: str) -> bool:
    parsed = urlparse(url)
    return (
        parsed.scheme == 'https'
        and parsed.netloc == 'graph.microsoft.com'
        and parsed.path.startswith(f'/beta/security/auditLog/queries/{qid}/records')
    )


def _validate_graph_records_url(url, qid: str) -> str:
    if not isinstance(url, str) or not url.strip():
        raise RuntimeError(f'Query {qid} returned missing/invalid @odata.nextLink: {url!r}')
    url = url.strip()
    if not _is_graph_records_url(url, qid):
        raise RuntimeError(f'Query {qid} returned non-Graph records continuation URL: {url}')
    return url


def _validate_records_page(qid: str, current_url: str, payload):
    if not isinstance(payload, dict):
        raise RuntimeError(
            f'Query {qid} returned malformed records payload type {type(payload).__name__} for {current_url}'
        )
    if 'value' not in payload:
        raise RuntimeError(f'Query {qid} records payload missing value list for {current_url}')
    vals = payload['value']
    if not isinstance(vals, list):
        raise RuntimeError(f'Query {qid} records payload value must be a list for {current_url}')
    for index, record in enumerate(vals):
        if not isinstance(record, dict):
            raise RuntimeError(
                f'Query {qid} records payload item {index} must be an object for {current_url}'
            )
    next_link = payload.get('@odata.nextLink')
    if next_link is None:
        return vals, None
    return vals, _validate_graph_records_url(next_link, qid)


def _drain(qid, key):
    n, page = 0, 0
    partial_files = []
    purge_window_files(STAGING_ABS, key, include_partial=True)
    url = _validate_graph_records_url(
        f'https://graph.microsoft.com/beta/security/auditLog/queries/{qid}/records?$top=999',
        qid,
    )
    seen_urls = set()
    while url:
        if url in seen_urls:
            raise RuntimeError(f'Query {qid} returned repeated pagination link cycle: {url}')
        seen_urls.add(url)
        r = _request('GET', url, headers=get_headers(), timeout=60)
        r.raise_for_status()
        vals, next_url = _validate_records_page(qid, url, r.json())
        if next_url in seen_urls:
            raise RuntimeError(f'Query {qid} returned repeated pagination link cycle: {next_url}')
        if vals:
            part_path = _stage_path_join(STAGING_ABS, f'win_{key}_{page:04d}.jsonl.partial')
            _write_stage_jsonl(STAGING_ABS, part_path, vals)
            partial_files.append(part_path)
            page += 1
            n += len(vals)
        url = next_url
    for part_path in partial_files:
        _move_stage_file(STAGING_ABS, part_path, part_path[:-8], overwrite=True)
    return n, page


# --- Audit sources -----------------------------------------------------------------------------
# AUDIT_SOURCE picks where each window's records come from. Every source stages records in the Graph
# audit record shape, so canonicalising, flattening and the MERGE by Id are the same whatever the source,
# and a record read by two sources is still one row.

class NoSourceForWindow(RuntimeError):
    """No enabled audit source holds this window's records."""


SENTINEL_QUERY_VIA = str(globals().get('SENTINEL_QUERY_VIA', 'hunting') or 'hunting').strip().lower()
SENTINEL_WORKSPACE_ID = str(globals().get('SENTINEL_WORKSPACE_ID', '') or '').strip()
SENTINEL_TABLE = str(globals().get('SENTINEL_TABLE', 'CopilotActivity') or 'CopilotActivity').strip()
SENTINEL_PAGE_ROWS = int(globals().get('SENTINEL_PAGE_ROWS', 2000))
SENTINEL_RECORD_TYPES = ('CopilotInteraction', '261')
FEED_HOST = str(globals().get('FEED_HOST', 'manage.office.com') or 'manage.office.com').strip().lower()
FEED_PUBLISHER_ID = str(globals().get('FEED_PUBLISHER_ID', '') or '').strip()
FEED_DOWNLOAD_THREADS = int(globals().get('FEED_DOWNLOAD_THREADS', 8))
FEED_RETENTION = timedelta(days=7)       # the Management Activity API lists content for 7 days
FEED_MARGIN = timedelta(minutes=15)      # windows start this far inside the 7 days
FEED_OVERLAP = timedelta(hours=6)        # each listing repeats the end of the last, for content listed late
FEED_STATE = _stage_path_join(STAGING_ABS, '_feed_state.json')
SOURCE_PLAN = {'sentinel_from': None, 'feed_floor': None}
PREFETCHED_KEYS = set()
AUDIT_STATUS_TABLE = 'dbo.audit_source_status'
SOURCE_LABELS = {'sentinel': 'Microsoft Sentinel', 'feed': 'the Office 365 Management Activity API',
                 'graph': 'the Graph audit search'}
SOURCE_STATUS = {}   # source -> ok, empty, skipped, forbidden or error, for the Load Status step
_MISSING_SOURCE = re.compile(r'failed to resolve|could not be resolved|not found|does not exist|PathNotFound', re.I)
_KQL_DATETIME = re.compile(r'^\d{4}-\d{2}-\d{2}T\d{2}:\d{2}:\d{2}(\.\d{1,7})?Z?$')
_KQL_IDENTIFIER = re.compile(r'^[A-Za-z_][A-Za-z0-9_]*$')
_USER_TYPES = {'regular': 0, 'reserved': 1, 'admin': 2, 'dcadmin': 3, 'system': 4, 'application': 5,
               'serviceprincipal': 6, 'custompolicy': 7, 'systempolicy': 8}


def pick_window_source(win_start, win_end, source, sentinel_from=None, feed_floor=None,
                       graph_for_history=False, write_mode='overwrite'):
    """The source for one window: 'sentinel', 'feed', 'graph', or None when no enabled source holds it.

    Sentinel holds the hours from its first Copilot record (sentinel_from). The Management Activity API
    holds the last 7 days (feed_floor). When Graph reads history (graph_for_history), a window that starts
    before the floor uses the API only when the run merges, because an earlier run read the part before
    the floor. Without Graph, a window straddling either start is read from the source that holds its end.
    """
    source = str(source or 'graph').strip().lower()
    if source == 'graph':
        return 'graph'
    graph_ok = bool(graph_for_history)
    ws, we = _as_utc_datetime(win_start), _as_utc_datetime(win_end)
    sentinel_from, feed_floor = _as_utc_datetime(sentinel_from), _as_utc_datetime(feed_floor)
    sentinel_on = source in ('auto', 'sentinel') and sentinel_from is not None
    if sentinel_on and ws >= sentinel_from:
        return 'sentinel'
    if (source in ('auto', 'feed') and feed_floor is not None and we > feed_floor
            and (ws >= feed_floor or write_mode == 'merge' or not graph_ok)):
        return 'feed'
    if graph_ok:
        return 'graph'
    if sentinel_on and we > sentinel_from:
        return 'sentinel'
    return None


def window_source(win_start, win_end):
    return pick_window_source(
        win_start, win_end, globals().get('AUDIT_SOURCE', 'graph'),
        SOURCE_PLAN.get('sentinel_from'), SOURCE_PLAN.get('feed_floor'),
        bool(globals().get('GRAPH_FOR_HISTORY', False)), globals().get('WRITE_MODE', 'overwrite'),
    )


def _utc_seconds(value):
    """'YYYY-MM-DDTHH:MM:SS' (UTC) of a datetime or an ISO timestamp, or None. Fractions are dropped."""
    if isinstance(value, datetime):
        return _as_utc_datetime(value).strftime('%Y-%m-%dT%H:%M:%S')
    text = str(value or '').strip()
    if text.endswith('+00:00'):
        text = text[:-6] + 'Z'
    return text[:19] if _KQL_DATETIME.match(text) else None


def _seconds_dt(value):
    text = _utc_seconds(value)
    return _as_utc_datetime(text + 'Z') if text else None


# --- Microsoft Sentinel (CopilotActivity) -------------------------------------------------------
def _kql_string(value) -> str:
    text = str(value).replace('\\', '\\\\').replace('"', '\\"').replace('\n', '\\n').replace('\r', '\\r')
    return f'"{text}"'


def _kql_datetime(value) -> str:
    """A KQL datetime literal. A string keeps its 7-digit fraction, so a page cursor matches exactly."""
    if isinstance(value, datetime):
        text = _as_utc_datetime(value).strftime('%Y-%m-%dT%H:%M:%S.%fZ')
    else:
        text = str(value or '').strip()
        if text.endswith('+00:00'):
            text = text[:-6] + 'Z'
    if not _KQL_DATETIME.match(text):
        raise ValueError(f'Not a UTC timestamp for a KQL query: {value!r}')
    return f'datetime({text if text.endswith("Z") else text + "Z"})'


def _sentinel_table() -> str:
    table = globals().get('SENTINEL_TABLE', 'CopilotActivity')
    if not isinstance(table, str) or not _KQL_IDENTIFIER.match(table):
        raise ValueError(f'SENTINEL_TABLE must be a table name; found {table!r}.')
    return table


def sentinel_page_query(win_start, win_end, after=None, rows=None) -> str:
    """KQL for one page of a window's Copilot records, oldest first. after = (TimeGenerated, RecordId) of
    the previous page's last row: a keyset cursor, so no page repeats or skips a row."""
    types = ', '.join(_kql_string(t) for t in SENTINEL_RECORD_TYPES)
    query = (f'{_sentinel_table()} | where TimeGenerated >= {_kql_datetime(win_start)} '
             f'and TimeGenerated < {_kql_datetime(win_end)} | where tostring(RecordType) in ({types})')
    if after:
        t, rid = _kql_datetime(after[0]), _kql_string(after[1])
        query += f' | where TimeGenerated > {t} or (TimeGenerated == {t} and strcmp(tostring(RecordId), {rid}) > 0)'
    return query + (f' | extend _RecordId = tostring(RecordId) | sort by TimeGenerated asc, _RecordId asc'
                    f' | take {max(int(rows or SENTINEL_PAGE_ROWS), 1)} | project-away _RecordId')


def _parse_dynamic(value):
    if isinstance(value, str) and value[:1] in ('{', '['):
        try:
            return json.loads(value)
        except ValueError:
            return value
    return value


def _is_size_limit_error(text) -> bool:
    text = str(text or '').lower()
    return any(word in text for word in ('limit', 'exceed', 'too large', 'size'))


def run_sentinel_query(query, start, end) -> list:
    """The rows of a KQL query as dicts: through Graph advanced hunting (SENTINEL_QUERY_VIA = 'hunting'),
    or the Log Analytics query API ('loganalytics'). A result over the service's size limit raises
    RecordLimitExceeded, so the window is split."""
    span = (f'{_as_utc_datetime(start):%Y-%m-%dT%H:%M:%SZ}/'
            f'{_as_utc_datetime(end) + timedelta(seconds=1):%Y-%m-%dT%H:%M:%SZ}')
    via = globals().get('SENTINEL_QUERY_VIA', 'hunting')
    workspace = globals().get('SENTINEL_WORKSPACE_ID', '')
    if via == 'loganalytics':
        if not workspace:
            raise ValueError("SENTINEL_QUERY_VIA = 'loganalytics' needs SENTINEL_WORKSPACE_ID (the Log Analytics workspace ID).")
        r = _request('POST', f'https://api.loganalytics.io/v1/workspaces/{workspace}/query',
                     headers=get_scoped_headers('https://api.loganalytics.io'),
                     json={'query': query, 'timespan': span}, timeout=300)
    elif via == 'hunting':
        body = {'Query': query, 'Timespan': span}
        if workspace:
            body['workspaceId'] = workspace
        r = _request('POST', 'https://graph.microsoft.com/v1.0/security/runHuntingQuery',
                     headers=get_headers(), json=body, timeout=300)
    else:
        raise ValueError(f"Unsupported SENTINEL_QUERY_VIA={via!r}; expected 'hunting' or 'loganalytics'.")
    if r.status_code == 400 and _is_size_limit_error(r.text):
        raise RecordLimitExceeded(f'The Sentinel query returned too much data: {r.text[:300]}')
    r.raise_for_status()
    payload = r.json() or {}
    if via == 'loganalytics':
        if payload.get('error'):
            detail = json.dumps(payload['error'])[:500]
            if _is_size_limit_error(detail):
                raise RecordLimitExceeded(f'The Log Analytics query returned too much data: {detail}')
            raise AuditQueryFailed(f'The Log Analytics query returned an error: {detail}')
        tables = payload.get('tables')
        if not isinstance(tables, list) or not tables or not isinstance(tables[0].get('rows'), list):
            raise AuditQueryFailed('The Log Analytics query returned no result table.')
        columns = [c['name'] for c in tables[0]['columns']]
        if any(not isinstance(row, list) or len(row) != len(columns) for row in tables[0]['rows']):
            raise AuditQueryFailed('The Log Analytics query returned an incomplete result row.')
        rows = [dict(zip(columns, row)) for row in tables[0]['rows']]
    else:
        rows = payload.get('results')
    if not isinstance(rows, list) or any(not isinstance(row, dict) for row in rows):
        raise AuditQueryFailed('The Sentinel query returned no valid result rows.')
    return [{k: _parse_dynamic(v) for k, v in row.items()} for row in rows]


def sentinel_first_record(start, end):
    """When the Sentinel table's first Copilot record in [start, end) is, or None when it has none."""
    types = ', '.join(_kql_string(t) for t in SENTINEL_RECORD_TYPES)
    query = (f'{_sentinel_table()} | where TimeGenerated >= {_kql_datetime(start)} and TimeGenerated < {_kql_datetime(end)}'
             f' | summarize FirstRecord = minif(TimeGenerated, tostring(RecordType) in ({types})),'
             f' RecordTypes = make_set(tostring(RecordType), 10)')
    rows = run_sentinel_query(query, start, end)
    row = rows[0] if rows else {}
    first = _seconds_dt(row.get('FirstRecord'))
    if first is None and row.get('RecordTypes'):
        print(f"  Note: {_sentinel_table()} has rows, but none with RecordType {' or '.join(SENTINEL_RECORD_TYPES)} "
              f"(found {row.get('RecordTypes')}).")
    return first


def sentinel_record(row) -> dict:
    """A CopilotActivity row as a Graph audit record, so it stages and parses like one."""
    row = row or {}
    when = _utc_seconds(row.get('TimeGenerated'))
    when = when + 'Z' if when else None
    event = row.get('LLMEventData')
    if isinstance(event, dict):
        event = dict(event)
        if row.get('AppHost') and not event.get('AppHost'):
            event['AppHost'] = row['AppHost']
        if row.get('AIModelName') and not event.get('ModelTransparencyDetails'):
            event['ModelTransparencyDetails'] = [{'ModelName': row['AIModelName'], 'ModelVersion': row.get('AIModelVersion')}]
    lifted = event if isinstance(event, dict) else {}
    user_type = row.get('ActorUserType')
    if isinstance(user_type, str):
        text = user_type.strip()
        user_type = int(text) if text.isdigit() else _USER_TYPES.get(text.lower().replace(' ', ''), text or None)
    audit = {
        'Id': row.get('RecordId'),
        'CreationTime': when,
        'Operation': 'CopilotInteraction',
        'RecordType': 261,
        'UserId': row.get('ActorName'),
        'UserKey': row.get('ActorUserId'),
        'UserType': user_type,
        'OrganizationId': row.get('OrganizationId'),
        'ClientIP': row.get('SrcIpAddr'),
        'ClientRegion': row.get('ClientRegion'),
        'Workload': row.get('Workload'),
        'Version': row.get('Version'),
        'AppIdentity': row.get('AppIdentity'),
        'AgentId': row.get('AgentId'),
        'AgentName': row.get('AgentName'),
        'AgentPlatform': row.get('AgentPlatform') or lifted.get('AgentPlatform'),
        'PlatformAgentId': row.get('PlatformAgentId') or lifted.get('PlatformAgentId'),
        'CopilotLogVersion': row.get('LogVersion'),
        'CopilotEventData': event,
    }
    audit = {k: v for k, v in audit.items() if v not in (None, '')}
    return {
        'id': row.get('RecordId'),
        'createdDateTime': when,
        'auditLogRecordType': 'CopilotInteraction',
        'operation': 'CopilotInteraction',
        'auditData': audit,
    }


def _drain_sentinel(key, win_start, win_end):
    n, page, after = 0, 0, None
    partial_files = []
    purge_window_files(STAGING_ABS, key, include_partial=True)
    while True:
        rows = run_sentinel_query(sentinel_page_query(win_start, win_end, after), win_start, win_end)
        if rows:
            part_path = _stage_path_join(STAGING_ABS, f'win_{key}_{page:04d}.jsonl.partial')
            _write_stage_jsonl(STAGING_ABS, part_path, [sentinel_record(r) for r in rows])
            partial_files.append(part_path)
            page += 1
            n += len(rows)
        if len(rows) < SENTINEL_PAGE_ROWS:
            break
        last = rows[-1]
        cursor = (last.get('TimeGenerated'), last.get('RecordId'))
        if not cursor[0] or not cursor[1] or cursor == after:
            raise RuntimeError(f'{key}: the Sentinel page cursor did not advance (last row {cursor!r})')
        after = cursor
    for part_path in partial_files:
        _move_stage_file(STAGING_ABS, part_path, part_path[:-8], overwrite=True)
    return n, page


def _process_sentinel(key, win_start, win_end, attempt=1):
    _mark_window(
        key,
        'querying',
        source='sentinel',
        query_id=None,
        window_start=_as_utc_datetime(win_start).isoformat(),
        window_end=_as_utc_datetime(win_end).isoformat(),
        refreshed_at=datetime.now(timezone.utc).isoformat(),
        attempts=attempt,
        error=None,
        **({'attempt_errors': []} if attempt == 1 else {}),
    )
    try:
        with QUERY_LIMITER:
            n, pages = _drain_sentinel(key, win_start, win_end)
        _mark_window(
            key,
            'succeeded',
            source='sentinel',
            rows=n,
            pages=pages,
            files=[_stage_basename(p) for p in list_window_files(STAGING_ABS, key)],
            completed_at=datetime.now(timezone.utc).isoformat(),
        )
        return key, n, 'done'
    except Exception as e:
        SOURCE_STATUS['sentinel'] = source_error_status(e)
        _mark_window(key, 'failed', error=f'{type(e).__name__}: {e}', completed_at=None)
        purge_window_files(STAGING_ABS, key, include_partial=True)
        raise


# --- Office 365 Management Activity API (the activity feed) -------------------------------------
def _feed_url(path, **params) -> str:
    if FEED_PUBLISHER_ID:
        params['PublisherIdentifier'] = FEED_PUBLISHER_ID
    query = ('?' + urlencode(params, safe=':')) if params else ''
    return f'https://{FEED_HOST}/api/v1.0/{TENANT_ID}/activity/feed/{path}{query}'


def _feed_headers() -> dict:
    return get_scoped_headers(f'https://{FEED_HOST}')


def _check_feed_url(url) -> str:
    parsed = urlparse(str(url or ''))
    if parsed.scheme != 'https' or parsed.netloc.lower() != FEED_HOST:
        raise RuntimeError(f'The Management Activity API returned a URL on another host: {url!r}')
    return url


def feed_ensure_subscription():
    """Start the Audit.General subscription unless it is on. Returns when this call started it, or None."""
    r = _request('GET', _feed_url('subscriptions/list'), headers=_feed_headers(), timeout=60)
    r.raise_for_status()
    for sub in r.json() or []:
        if (isinstance(sub, dict) and str(sub.get('contentType')).lower() == 'audit.general'
                and str(sub.get('status')).lower() == 'enabled'):
            return None
    r = _request('POST', _feed_url('subscriptions/start', contentType='Audit.General'), headers=_feed_headers(), timeout=60)
    r.raise_for_status()
    return datetime.now(timezone.utc)


def feed_list_content(start, end) -> list:
    """The Audit.General content published in [start, end), listed in slices of at most 24 hours."""
    blobs, cur, end = [], _as_utc_datetime(start), _as_utc_datetime(end)
    while cur < end:
        stop = min(cur + timedelta(hours=24), end)
        url = _feed_url('subscriptions/content', contentType='Audit.General',
                        startTime=f'{cur:%Y-%m-%dT%H:%M:%S}', endTime=f'{stop:%Y-%m-%dT%H:%M:%S}')
        visited = set()
        while url:
            if url in visited:
                raise RuntimeError(f'The Management Activity API repeated a content page: {url}')
            visited.add(url)
            r = _request('GET', url, headers=_feed_headers(), timeout=60)
            r.raise_for_status()
            page = r.json() or []
            if not isinstance(page, list):
                raise RuntimeError(f'The Management Activity API returned a {type(page).__name__} content list for {url}')
            blobs.extend(b for b in page if isinstance(b, dict) and b.get('contentUri'))
            nxt = r.headers.get('NextPageUri')
            url = _check_feed_url(nxt) if nxt else None
        cur = stop
    return blobs


def feed_download(content_uri) -> list:
    """The Copilot interaction records (RecordType 261) in one content blob."""
    r = _request('GET', _check_feed_url(content_uri), headers=_feed_headers(), timeout=120)
    r.raise_for_status()
    data = r.json()
    if not isinstance(data, list):
        raise RuntimeError(f'The Management Activity API returned a {type(data).__name__} for {content_uri}')
    # RecordType comes back as a number or a string.
    return [rec for rec in data if isinstance(rec, dict) and str(rec.get('RecordType')).strip() == '261']


def feed_record(record) -> dict:
    """A Management Activity API record as a Graph audit record: the same Id, times and AuditData."""
    audit = dict(record or {})
    created = _utc_seconds(audit.get('CreationTime'))
    if created:
        audit['CreationTime'] = created + 'Z'
    return {
        'id': audit.get('Id'),
        'createdDateTime': created + 'Z' if created else None,
        'auditLogRecordType': 'CopilotInteraction',
        'operation': audit.get('Operation'),
        'auditData': audit,
    }


def bucket_feed_records(records, leaves) -> dict:
    """Records (Graph-shaped) grouped by the window holding their time; records in no window are dropped."""
    spans = sorted((_as_utc_datetime(ws), _as_utc_datetime(we), stable_window_key(ws, we)) for ws, we in leaves)
    starts = [s for s, _, _ in spans]
    out = {key: [] for _, _, key in spans}
    for rec in records:
        when = _as_utc_datetime((rec or {}).get('createdDateTime'))
        i = bisect.bisect_right(starts, when) - 1 if when is not None else -1
        if i >= 0 and when < spans[i][1]:
            out[spans[i][2]].append(rec)
    return out


def load_feed_state() -> dict:
    if not _stage_exists(FEED_STATE):
        return {}
    try:
        data = _read_stage_json(STAGING_ABS, FEED_STATE)
    except (ValueError, UnicodeDecodeError) as e:
        print(f'  {FEED_STATE} could not be read ({e}); the Management Activity API windows are read in full.')
        return {}
    return data if isinstance(data, dict) else {}


def save_feed_state(state, now):
    """Save the listing state. Content IDs older than the API keeps are dropped."""
    cutoff = now - FEED_RETENTION - timedelta(days=1)
    seen = {k: v for k, v in (state.get('seen') or {}).items() if (_seconds_dt(v) or now) >= cutoff}
    payload = {**state, 'seen': seen}
    temp_path = _stage_path_join(STAGING_ABS, f'._feed_state.{os.getpid()}.{threading.get_ident()}.tmp')
    try:
        _write_stage_text(STAGING_ABS, temp_path, json.dumps(payload, indent=1, sort_keys=True))
        _move_stage_file(STAGING_ABS, temp_path, FEED_STATE, overwrite=True)
    finally:
        if _stage_exists(temp_path):
            _stage_remove(temp_path)


def prefetch_feed(leaves, now=None) -> set:
    """Fill every window the Management Activity API reads, from one listing of its content.

    The API lists content by when it was published, not when the activity happened, so one listing
    serves every window: each record goes to the window holding its CreationTime. A window this run
    has to read in full is rebuilt from all content published since it started. A window an earlier run
    filled from the API only gets the records in content published since that run. Records outside these
    windows are dropped. Returns the keys of the windows filled.
    """
    now = now or datetime.now(timezone.utc)
    trailing_cutoff = end_date - timedelta(days=max(int(LOOKBACK_DAYS), 0))
    fresh_after = trailing_fresh_after(now, TRAILING_REUSE_HOURS)
    full, topup = [], []
    for ws, we in leaves:
        if window_source(ws, we) != 'feed':
            continue
        key = stable_window_key(ws, we)
        entry = _read_manifest_entry(key)
        if is_window_reusable(STAGING_ABS, key, we, entry, trailing_cutoff, fresh_after):
            continue
        if (manifest_succeeded(entry) and entry.get('source') == 'feed'
                and window_stage_files_reusable(STAGING_ABS, key, entry)):
            topup.append((ws, we))
        else:
            full.append((ws, we))
    if not full and not topup:
        return set()

    state = load_feed_state()
    seen = dict(state.get('seen') or {})
    full_from = min((_as_utc_datetime(ws) for ws, _ in full), default=None)
    full_cut = full_from - timedelta(hours=1) if full_from is not None else None
    listed_through = _seconds_dt(state.get('listed_through'))
    starts = [full_cut] if full_cut is not None else []
    starts.append(listed_through - FEED_OVERLAP if listed_through
                  else min(_as_utc_datetime(ws) for ws, _ in full + topup) - timedelta(hours=1))
    list_from = max(min(starts), now - FEED_RETENTION + timedelta(minutes=1))
    blobs = feed_list_content(list_from, now)
    if any(not b.get('contentId') or not b.get('contentUri') for b in blobs):
        raise AuditQueryFailed('The Management Activity API returned content without an ID or download URL.')
    blobs = list({b['contentId']: b for b in blobs}.values())
    wanted = [b for b in blobs if b.get('contentId') not in seen
              or (full_cut is not None and (_seconds_dt(b.get('contentCreated')) or now) >= full_cut)]

    def _download(blob):
        try:
            return blob, feed_download(blob['contentUri'])
        except Exception as e:
            if _http_status(e) == 404:
                raise AuditQueryFailed(
                    f"Content {blob.get('contentId')} expired before it was read; its records are missing. "
                    'The listing checkpoint was not advanced; rerun the load.'
                ) from e
            raise

    with ThreadPoolExecutor(max_workers=max(1, FEED_DOWNLOAD_THREADS)) as ex:
        results = list(ex.map(_download, wanted))
    fresh, again = [], []
    for blob, records in results:
        # Content read by an earlier run only feeds the windows read in full; the others already hold it.
        (again if blob.get('contentId') in seen else fresh).extend(feed_record(r) for r in records)
    buckets = bucket_feed_records(fresh, full + topup)
    for key, records in bucket_feed_records(again, full).items():
        buckets[key].extend(records)

    full_keys = {stable_window_key(ws, we) for ws, we in full}
    stamp = now.strftime('%Y%m%d%H%M%S')
    filled = set()
    for ws, we in sorted(full + topup):
        key = stable_window_key(ws, we)
        records = buckets.get(key) or []
        if key in full_keys:
            purge_window_files(STAGING_ABS, key, include_partial=True)
        if records:
            part_path = _stage_path_join(STAGING_ABS, f'win_{key}_feed{stamp}.jsonl.partial')
            _write_stage_jsonl(STAGING_ABS, part_path, records)
            _move_stage_file(STAGING_ABS, part_path, part_path[:-8], overwrite=True)
        previous = 0 if key in full_keys else int(_read_manifest_entry(key).get('rows', 0) or 0)
        files = [_stage_basename(p) for p in list_window_files(STAGING_ABS, key)]
        _mark_window(
            key,
            'succeeded',
            source='feed',
            query_id=None,
            window_start=_as_utc_datetime(ws).isoformat(),
            window_end=_as_utc_datetime(we).isoformat(),
            refreshed_at=now.isoformat(),
            attempts=1,
            error=None,
            rows=previous + len(records),
            pages=len(files),
            files=files,
            completed_at=now.isoformat(),
        )
        filled.add(key)
    for blob in blobs:
        if blob.get('contentId'):
            seen[blob['contentId']] = blob.get('contentCreated')
    state.update(listed_through=f'{now:%Y-%m-%dT%H:%M:%S}Z', seen=seen)
    save_feed_state(state, now)
    SOURCE_STATUS['feed'] = 'ok' if fresh or again else 'empty'
    print(f'Management Activity API: listed {len(blobs)} content blob(s) published since '
          f'{list_from:%Y-%m-%d %H:%M} UTC, read {len(wanted)}; '
          f'{sum(len(v) for v in buckets.values()):,} Copilot record(s) for {len(filled)} window(s) '
          f'({len(full)} read in full, {len(topup)} topped up).')
    return filled


def source_error_status(exc) -> str:
    """How a source that couldn't be read is recorded: 'forbidden' when sign-in or a permission was refused,
    'skipped' when it isn't there (no Sentinel, or no CopilotActivity table), otherwise 'error'."""
    response = getattr(exc, 'response', None)
    text = f"{exc} {getattr(response, 'text', '') or ''}"
    if _http_status(exc) in (401, 403) or re.search(r'forbidden|unauthori[sz]ed|AADSTS\d+|insufficient privileges', text, re.I):
        return 'forbidden'
    if _MISSING_SOURCE.search(text):
        return 'skipped'
    return 'error'


def source_status_rows(statuses, run_at) -> list:
    """The audit_source_status rows: one per source this run tried, by its plain name."""
    return [{'RunAt': run_at, 'Source': SOURCE_LABELS.get(k, k), 'Status': v}
            for k, v in statuses.items() if v]


def write_source_status(now=None):
    """Replace audit_source_status with this run's sources, so the Load Status step can mark a run that
    loaded but couldn't read a source as a Warning. A table it can't write never fails the load."""
    rows = source_status_rows(SOURCE_STATUS, now or datetime.now(timezone.utc))
    try:
        (spark.createDataFrame(rows, 'RunAt timestamp, Source string, Status string').write
            .format('delta').mode('overwrite').option('overwriteSchema', 'true')
            .saveAsTable(AUDIT_STATUS_TABLE))
    except Exception as e:  # noqa: BLE001 - only the Load Status warning depends on it
        print(f'Note: could not write {AUDIT_STATUS_TABLE} ({type(e).__name__}: {e}).')


def plan_audit_sources(planned, now=None) -> dict:
    """Find the hours Sentinel and the Management Activity API hold, for window_source.

    Sentinel holds the hours from its first Copilot record. The API holds the last 7 days, or the time
    since this notebook started its subscription. A source that can't be read is skipped with a warning
    under 'auto'; when it is the only source, the run stops with the fix.
    """
    now = now or datetime.now(timezone.utc)
    source = globals().get('AUDIT_SOURCE', 'graph')
    plan = {'sentinel_from': None, 'feed_floor': None}
    if source == 'graph' or not planned:
        if source == 'graph':
            SOURCE_STATUS['graph'] = 'ok'
        return plan
    first = min(_as_utc_datetime(ws) for ws, _ in planned)
    if globals().get('GRAPH_FOR_HISTORY', False):
        SOURCE_STATUS['graph'] = 'ok'
    if source in ('auto', 'sentinel'):
        since = first - timedelta(days=max(int(globals().get('EMPTY_RECHECK_DAYS', 0) or 0), 0) + 1)
        try:
            plan['sentinel_from'] = sentinel_first_record(since, now)
            SOURCE_STATUS['sentinel'] = 'ok' if plan['sentinel_from'] is not None else 'empty'
        except Exception as e:
            SOURCE_STATUS['sentinel'] = source_error_status(e)
            fix = ("give the app the Microsoft Graph application permission ThreatHunting.Read.All with admin consent"
                   if globals().get('SENTINEL_QUERY_VIA', 'hunting') == 'hunting' else
                   'give the app the Log Analytics Reader role on the workspace, and check SENTINEL_WORKSPACE_ID')
            message = f'Microsoft Sentinel could not be read ({type(e).__name__}: {e}). Fix: {fix}.'
            if source == 'sentinel':
                raise RuntimeError(message) from e
            print(f'WARNING: {message} Sentinel is skipped for this run.')
        if plan['sentinel_from'] is not None:
            print(f"Sentinel {_sentinel_table()} holds Copilot records from {plan['sentinel_from']:%Y-%m-%d %H:%M} UTC.")
        elif source == 'sentinel':
            raise RuntimeError(
                f'Microsoft Sentinel table {_sentinel_table()} has no Copilot records since {since:%Y-%m-%d}. '
                f'Connect the Microsoft Copilot data connector in Sentinel (Content hub -> Microsoft Copilot), '
                f"or set AUDIT_SOURCE = 'feed' or 'auto'."
            )
    if source in ('auto', 'feed'):
        floor = now - FEED_RETENTION + FEED_MARGIN
        write_mode = globals().get('WRITE_MODE', 'overwrite')
        if any(pick_window_source(ws, we, source, plan['sentinel_from'], floor, False, write_mode) == 'feed'
               for ws, we in leaf_windows(planned)):
            try:
                state = load_feed_state()
                started = feed_ensure_subscription()
                if started is not None:
                    state['subscription_started_at'] = started.isoformat()
                    save_feed_state(state, now)
                    SOURCE_STATUS['feed'] = 'empty'
                    print('Started the Management Activity API subscription (Audit.General). Its first content '
                          'can take up to 12 hours to appear; later runs read it.')
                started_at = _as_utc_datetime(state.get('subscription_started_at'))
                plan['feed_floor'] = floor if started_at is None else max(floor, started_at)
                print(f"Management Activity API holds records from {plan['feed_floor']:%Y-%m-%d %H:%M} UTC.")
            except Exception as e:
                SOURCE_STATUS['feed'] = source_error_status(e)
                if SOURCE_STATUS['feed'] == 'skipped':
                    SOURCE_STATUS['feed'] = 'error'
                message = (f'The Office 365 Management Activity API could not be read ({type(e).__name__}: {e}). '
                           f'Fix: give the app the Office 365 Management APIs application permission '
                           f'ActivityFeed.Read with admin consent, and check FEED_HOST.')
                if not globals().get('GRAPH_FOR_HISTORY', False):
                    raise RuntimeError(message) from e
                print(f'WARNING: {message} Its windows are read from Graph instead.')
    return plan


def drop_out_of_reach(planned, write_mode):
    """Leave out the windows no enabled source holds, and keep the history already in the table.

    Without Graph, Sentinel and the Management Activity API hold only recent hours (the API: 7 days),
    so older windows are left out; GRAPH_FOR_HISTORY = True loads them once from Graph. When a window is
    left out, or starts before the hours the sources hold, an overwrite would delete history this run
    can't reload, so the run merges instead. Returns (windows to read, windows left out, write mode).
    """
    source = str(globals().get('AUDIT_SOURCE', 'graph') or 'graph').strip().lower()
    if source == 'graph' or globals().get('GRAPH_FOR_HISTORY', False):
        return list(planned), [], write_mode
    kept, dropped = [], []
    for ws, we in planned:
        (kept if window_source(ws, we) else dropped).append((ws, we))
    held = [_as_utc_datetime(v) for v in (SOURCE_PLAN.get('sentinel_from'), SOURCE_PLAN.get('feed_floor'))
            if v is not None]
    reach = min(held) if held else None
    partial = reach is not None and any(_as_utc_datetime(ws) < reach for ws, _ in kept)
    if write_mode in ('replace', 'overwrite') and (dropped or partial):
        write_mode = 'merge'
    return kept, dropped, write_mode


def _window_fetch_source(win_start, win_end):
    """The source fetch_windows reads a window from. The Management Activity API is read up front
    (prefetch_feed), so a 'feed' window reaching here (a split, or the API failed) falls back to Graph."""
    if globals().get('AUDIT_SOURCE', 'graph') == 'graph':
        return 'graph'
    source = window_source(win_start, win_end)
    if source == 'feed':
        source = 'graph' if globals().get('GRAPH_FOR_HISTORY', False) else None
    return source


def _process(win, attempt=1, recheck=False):
    ws, we = win
    key = stable_window_key(ws, we)
    entry = _read_manifest_entry(key)
    trailing_cutoff = end_date - timedelta(days=max(int(LOOKBACK_DAYS), 0))
    now = datetime.now(timezone.utc)
    if not recheck and is_window_reusable(STAGING_ABS, key, we, entry, trailing_cutoff, trailing_fresh_after(now, TRAILING_REUSE_HOURS)):
        return key, entry.get('rows', 0), 'skip'
    if budget_exhausted(RUN_DEADLINE):
        raise BudgetReached('Time budget reached before this window started')
    source = _window_fetch_source(ws, we)
    if source is None:
        message = (
            f"no enabled audit source holds {_fmt_range(ws, we)} (AUDIT_SOURCE={globals().get('AUDIT_SOURCE')!r}). "
            f"Set GRAPH_FOR_HISTORY = True to read it from Graph audit search, or set "
            f'ALLOW_UNRECOVERED_GAPS = True to leave it empty.'
        )
        _mark_window(key, 'failed', source=None, window_start=_as_utc_datetime(ws).isoformat(),
                     window_end=_as_utc_datetime(we).isoformat(), error=f'NoSourceForWindow: {message}',
                     completed_at=None)
        raise NoSourceForWindow(message)
    if source == 'sentinel':
        return _process_sentinel(key, ws, we, attempt)
    resume_qid = resumable_query_id(entry, now, RESUME_QUERY_HOURS) if attempt == 1 else None
    if QUOTA_PAUSED.is_set() and not resume_qid:
        raise QueryQuotaReached('The audit search limit was reached earlier in this run')
    _mark_window(
        key,
        'querying',
        source='graph',
        window_start=_as_utc_datetime(ws).isoformat(),
        window_end=_as_utc_datetime(we).isoformat(),
        refreshed_at=datetime.now(timezone.utc).isoformat(),
        attempts=attempt,
        error=None,
        **({'attempt_errors': []} if attempt == 1 else {}),
    )
    qid, payload = None, None
    try:
        # Only creating and waiting take one of the tenant's concurrent query slots; draining doesn't.
        if resume_qid:
            qid = resume_qid
            try:
                with QUERY_LIMITER:
                    _mark_window(key, 'waiting', query_id=qid)
                    payload = wait_for_query(qid)
                print(f'    {key}: picked up query {qid} from an earlier attempt')
            except (BudgetReached, QueryQuotaReached):
                raise
            except Exception as exc:
                if _http_status(exc) in _FATAL_HTTP:
                    raise
                print(f'    {key}: query {qid} from an earlier attempt could not be used ({type(exc).__name__}); sending a new one')
                qid = None
        if qid is None:
            with QUERY_LIMITER:
                if QUOTA_PAUSED.is_set():
                    raise QueryQuotaReached('The audit search limit was reached earlier in this run')
                qid = create_query(ws, we)
                _mark_window(key, 'waiting', query_id=qid, query_created_at=datetime.now(timezone.utc).isoformat())
                payload = wait_for_query(qid)
        capped = isinstance(payload, dict) and payload.get('isRecordCountLimitExceeded') is True
        if capped and split_window(ws, we, MIN_CHUNK_HOURS):
            raise RecordLimitExceeded(f'Query {qid} reached the audit service\'s record limit')
        if capped:
            print(f'    WARNING {key}: query {qid} reached the audit service\'s record limit at the smallest '
                  f'window ({MIN_CHUNK_HOURS:g}h); some of its records are missing')
        _mark_window(key, 'draining', query_id=qid)
        n, pages = _drain(qid, key)
        _mark_window(
            key,
            'succeeded',
            query_id=qid,
            rows=n,
            pages=pages,
            files=[_stage_basename(p) for p in list_window_files(STAGING_ABS, key)],
            completed_at=datetime.now(timezone.utc).isoformat(),
            **({'record_limit_exceeded': True} if capped else {}),
        )
        return key, n, 'done'
    except (BudgetReached, QueryQuotaReached) as e:
        # Keep the query id: the retry re-polls the query instead of queueing a new one.
        stopped = datetime.now(timezone.utc).isoformat()
        _mark_window(key, 'waiting', paused_at=stopped, query_id=qid,
                     **({'throttled_at': stopped} if isinstance(e, QueryQuotaReached) else {}))
        raise
    except Exception as e:
        failure = {
            'error': f'{type(e).__name__}: {e}',
            'completed_at': None,
        }
        if qid:
            failure['query_id'] = qid
        _mark_window(key, 'failed', **failure)
        purge_window_files(STAGING_ABS, key, include_partial=True)
        raise


_FATAL_HTTP = {401, 403}


def _http_status(exc):
    status = getattr(getattr(exc, 'response', None), 'status_code', None)
    return status if isinstance(status, int) else None


def is_retryable_window_error(exc) -> bool:
    """Failures a fresh query can fix: failed/cancelled by the service, timeouts, throttling, outages."""
    if isinstance(exc, (AuditQueryFailed, TimeoutError, ConnectionError)):
        return True
    status = _http_status(exc)
    if status is not None:
        return status in _TRANSIENT
    return type(exc).__module__.split('.')[0] in ('requests', 'urllib3')


def _run_window(win, recheck=False):
    """Fetch one window: retry with a new query, then split it when the retries run out.

    Returns (key, rows, outcome, detail). outcome is 'skip', 'done', 'split' (detail = the halves),
    'paused' (TIME_BUDGET_MIN ran out), 'throttled' (Graph's audit search limit) or 'failed'
    (detail = the last error). Raises only for 401/403, which would fail every window.
    recheck: query the window even though it is reusable (an empty window due one more query).
    """
    ws, we = win
    key = stable_window_key(ws, we)
    retries = max(int(WINDOW_RETRIES), 0)
    errors, last = [], None
    for attempt in range(1, retries + 2):
        try:
            _, rows, how = _process(win, attempt=attempt, recheck=recheck)
            return key, rows, how, None
        except BudgetReached:
            return key, 0, 'paused', None
        except QueryQuotaReached:
            QUOTA_PAUSED.set()
            return key, 0, 'throttled', None
        except Exception as exc:
            if _http_status(exc) in _FATAL_HTTP:
                raise
            last = f'{type(exc).__name__}: {exc}'
            errors.append(f'attempt {attempt}: {last}')
            _mark_window(key, 'failed', attempts=attempt, attempt_errors=errors[-5:])
            if isinstance(exc, RecordLimitExceeded):
                break
            if not is_retryable_window_error(exc):
                return key, 0, 'failed', last
            if QUERY_LIMITER.shrink(f'{key}: {last}', halve=_http_status(exc) == 429):
                print(f'    concurrency lowered to {QUERY_LIMITER.limit} after: {last}')
            if isinstance(exc, TimeoutError) and SPLIT_ON_TIMEOUT:
                break
            if budget_exhausted(RUN_DEADLINE):
                return key, 0, 'paused', None
            if attempt <= retries:
                delay = retry_delay(attempt, RETRY_BASE_SEC, RETRY_MAX_SEC)
                print(f'    {key}: {last}; new query in {delay:.0f}s (retry {attempt}/{retries})')
                try:
                    _wait(delay)
                except BudgetReached:
                    return key, 0, 'paused', None
    halves = split_window(ws, we, MIN_CHUNK_HOURS)
    if not halves:
        return key, 0, 'failed', last
    _mark_window(
        key,
        'split',
        parts=[stable_window_key(h_ws, h_we) for h_ws, h_we in halves],
        split_at=datetime.now(timezone.utc).isoformat(),
        error=last,
        completed_at=None,
    )
    purge_window_files(STAGING_ABS, key, include_partial=True)
    return key, 0, 'split', halves


def leaf_windows(planned):
    """The windows that hold the data for a plan: split windows are replaced by their parts."""
    with MANIFEST_LOCK:
        snapshot = dict(manifest)
    return [leaf for ws, we in planned for leaf in expand_split_windows(snapshot, ws, we, MIN_CHUNK_HOURS)]


def _fmt_range(ws, we):
    return f'{_as_utc_datetime(ws):%Y-%m-%d %H:%M} -> {_as_utc_datetime(we):%Y-%m-%d %H:%M} UTC'


def assert_no_unrecovered_gaps(not_before, before):
    gaps = uncovered_failed_ranges(manifest, not_before, before)
    if gaps:
        lines = '\n'.join(f'  - {_fmt_range(s, e)}' for s, e in gaps)
        raise RuntimeError(
            f'{len(gaps)} earlier audit range(s) failed and were never recovered:\n{lines}\n'
            f'This incremental run starts at {_as_utc_datetime(before):%Y-%m-%d %H:%M} UTC, so it would leave '
            f"them empty. Rerun with MODE='backfill' first: succeeded windows are reused, so only these ranges "
            f'are queried again. To accept the gap instead, set ALLOW_UNRECOVERED_GAPS = True.'
        )


def plan_empty_rechecks(planned):
    """Empty windows due one more query (empty_recheck_windows), oldest first, at most EMPTY_RECHECK_MAX,
    plus every window still flagged RECHECK_PENDING by a run that stopped before writing.

    Returns the plan with those windows before it added, and the keys to query again even though they
    are reusable. Those keys stay flagged until clear_recheck_pending runs after the write, so a retry
    plans them again and writes what they fetched. A window that overlaps the plan without being one of
    its windows is left. A backfill adds no older windows: it writes only its own range.
    """
    with MANIFEST_LOCK:
        snapshot = dict(manifest)
    trailing_cutoff = end_date - timedelta(days=max(int(LOOKBACK_DAYS), 0))
    pending = pending_recheck_windows(snapshot)
    due = [w for w in empty_recheck_windows(snapshot, end_date, trailing_cutoff, EMPTY_RECHECK_DAYS, LOOKBACK_DAYS)
           if w not in pending]
    if globals().get('AUDIT_SOURCE', 'graph') != 'graph':
        # Another query of an empty window is cheap only in Sentinel; Graph is slow and the API forgets after 7 days.
        due = [w for w in due if window_source(*w) == 'sentinel']
    planned_keys = {stable_window_key(ws, we) for ws, we in leaf_windows(planned)}
    plan_start = min((_as_utc_datetime(ws) for ws, _ in planned), default=None)

    def fits(ws):
        return [w for w in ws if stable_window_key(*w) in planned_keys
                or (WRITE_MODE == 'merge' and (plan_start is None or w[1] <= plan_start))]

    kept = fits(pending)
    picked = fits(due)[:max(int(EMPTY_RECHECK_MAX), 0)]
    recheck = {stable_window_key(*w) for w in picked}
    if recheck:
        with MANIFEST_LOCK:
            current = dict(manifest)
            for key in recheck:
                current[key] = {**current.get(key, {}), RECHECK_PENDING: True}
            _save_manifest(current)
            manifest.clear()
            manifest.update(current)
    extra = [w for w in sorted(kept + picked) if stable_window_key(*w) not in planned_keys]
    return extra + list(planned), recheck


def clear_recheck_pending():
    """The run's rows are written: drop the RECHECK_PENDING flags."""
    with MANIFEST_LOCK:
        current = dict(manifest)
        flagged = [k for k, e in current.items() if isinstance(e, dict) and e.get(RECHECK_PENDING)]
        if not flagged:
            return
        for k in flagged:
            current[k] = {f: v for f, v in current[k].items() if f != RECHECK_PENDING}
        _save_manifest(current)
        manifest.clear()
        manifest.update(current)


def fetch_windows(planned, recheck=()):
    """Fetch every planned window (retrying and splitting); return (active keys, staged rows).

    recheck: keys of empty windows to query once more even though they are reusable.
    """
    recheck = set(recheck or ())
    trailing_cutoff = end_date - timedelta(days=max(int(LOOKBACK_DAYS), 0))
    fresh_after = trailing_fresh_after(datetime.now(timezone.utc), TRAILING_REUSE_HOURS)
    leaves = leaf_windows(planned)
    prefetched = set(globals().get('PREFETCHED_KEYS') or ()) & {stable_window_key(ws, we) for ws, we in leaves}
    pending = [
        (ws, we) for ws, we in leaves
        if stable_window_key(ws, we) not in prefetched and (stable_window_key(ws, we) in recheck or not is_window_reusable(
            STAGING_ABS, stable_window_key(ws, we), we, _read_manifest_entry(stable_window_key(ws, we)),
            trailing_cutoff, fresh_after,
        ))
    ]
    print(
        f'{len(leaves) - len(pending) - len(prefetched)} window(s) already reusable; '
        + (f'{len(prefetched)} read from the Management Activity API; ' if prefetched else '')
        + f'{len(pending)} to fetch '
        f'(<= {QUERY_LIMITER.limit} queries at a time; a failing window is retried {WINDOW_RETRIES}x, '
        f'then split down to {MIN_CHUNK_HOURS:g}h).'
        + (f' {len(recheck)} of them came back empty before and are queried once more.' if recheck else '')
    )
    outcomes, fatal, done, jobs = {}, None, 0, len(pending)
    with ThreadPoolExecutor(max_workers=max(1, int(MAX_CONCURRENT_QUERIES))) as ex:
        running = {ex.submit(_run_window, w, stable_window_key(*w) in recheck): w for w in pending}
        while running:
            finished, _ = wait_futures(list(running), return_when=FIRST_COMPLETED)
            for fut in finished:
                running.pop(fut)
                if fut.cancelled():
                    continue
                try:
                    key, n, how, detail = fut.result()
                except Exception as e:
                    fatal = fatal or e
                    for other in running:
                        other.cancel()
                    continue
                outcomes[key] = how
                done += 1
                if how == 'split':
                    hours = (detail[0][1] - detail[0][0]).total_seconds() / 3600
                    print(f'  [{done}/{jobs}] {key} kept failing or reached the record limit -> split into {len(detail)} x {hours:g}h')
                    if fatal is None:
                        for half in detail:
                            running[ex.submit(_run_window, half)] = half
                            jobs += 1
                elif how == 'failed':
                    print(f'  [{done}/{jobs}] {key} FAILED: {detail}')
                elif how == 'paused':
                    print(f'  [{done}/{jobs}] {key} left for the retry (time budget reached)')
                elif how == 'throttled':
                    print(f'  [{done}/{jobs}] {key} left for the next run (audit search limit reached)')
                else:
                    print(f'  [{done}/{jobs}] {key} {how} (+{n:,})')
    if fatal is not None:
        raise fatal

    leaves = leaf_windows(planned)
    with MANIFEST_LOCK:
        snapshot = dict(manifest)
    keys = [stable_window_key(ws, we) for ws, we in leaves]
    failed = [(w, snapshot.get(k) or {}) for w, k in zip(leaves, keys) if not manifest_succeeded(snapshot.get(k))]
    fetched = [k for k in keys if outcomes.get(k) == 'done']
    retried = [k for k in fetched if int((snapshot.get(k) or {}).get('attempts', 1) or 1) > 1]
    split = sum(1 for how in outcomes.values() if how == 'split')
    rows = sum(int((snapshot.get(k) or {}).get('rows', 0) or 0) for k in keys if manifest_succeeded(snapshot.get(k)))
    shrinks = ', '.join(f'{old}->{new}' for old, new, _ in QUERY_LIMITER.history) or 'unchanged'
    print(
        f'Summary: {len(planned)} planned window(s) -> {len(leaves)} after splits | '
        f'reused {len(leaves) - len(fetched) - len(failed) - len(prefetched)}, '
        + (f'Management Activity API {len(prefetched)}, ' if prefetched else '')
        + f'fetched {len(fetched)} ({len(retried)} after a retry), '
        f'split {split}, failed {len(failed)} | concurrency {shrinks} | staged records {rows:,}'
    )
    paused = [k for k, how in outcomes.items() if how == 'paused']
    throttled = [k for k, how in outcomes.items() if how == 'throttled']
    if throttled:
        raise RuntimeError(
            f'Microsoft Graph refused new audit searches for this tenant (HTTP 429) with {len(throttled) + len(paused)} '
            f'audit window(s) still to fetch ({len(fetched)} fetched this run). A tenant can start at least 200 audit '
            f'searches in a rolling 24 hours, shared with other admins and tools. Nothing was written to {OUTPUT_TABLE}. '
            f'The next run, or rerunning with the same MODE later, carries on from here: finished windows are reused '
            f'and queries still running are picked up again.'
        )
    if paused:
        raise RuntimeError(
            f'Time budget of {TIME_BUDGET_MIN} min reached with {len(paused)} audit window(s) still to fetch '
            f'({len(fetched)} fetched this run). Nothing was written to {OUTPUT_TABLE}. The pipeline retry, or rerunning with the same MODE, carries on from here: '
            f'finished windows are reused and queries still running are picked up again.'
        )
    gaps = [(w, entry) for w, entry in failed if str(entry.get('error') or '').startswith('NoSourceForWindow')]
    if gaps:
        lines = '\n'.join(f'  - {_fmt_range(ws, we)} [{stable_window_key(ws, we)}]' for (ws, we), _ in gaps)
        if not ALLOW_UNRECOVERED_GAPS:
            raise RuntimeError(
                f"{len(gaps)} audit window(s) are not held by any enabled source "
                f"(AUDIT_SOURCE={globals().get('AUDIT_SOURCE')!r}). Nothing was written to {OUTPUT_TABLE}.\n{lines}\n"
                f"Next step: set GRAPH_FOR_HISTORY = True to read them from Graph audit search, or "
                f'ALLOW_UNRECOVERED_GAPS = True to leave them empty.'
            )
        print(f'WARNING: {len(gaps)} audit window(s) are left empty (no enabled source holds them; '
              f'ALLOW_UNRECOVERED_GAPS = True):\n{lines}')
        failed = [f for f in failed if f not in gaps]
    if failed:
        lines = '\n'.join(
            f'  - {_fmt_range(ws, we)} [{stable_window_key(ws, we)}]: {entry.get("error") or entry.get("status")}'
            for (ws, we), entry in failed
        )
        rerun = "rerun with MODE='backfill'" if MODE == 'backfill' else 'rerun the notebook'
        raise RuntimeError(
            f'{len(failed)} audit window(s) still failed after {WINDOW_RETRIES} retries and splitting down to '
            f'{MIN_CHUNK_HOURS:g}h. Nothing was written to {OUTPUT_TABLE}.\n{lines}\n'
            f'Next step: {rerun}; succeeded windows are reused, so only the windows above are queried again. '
            f'If they keep failing, check for other audit searches running in the tenant and lower '
            f'MAX_CONCURRENT_QUERIES.'
        )
    if keys and rows == 0:
        first, last = min(ws for ws, _ in leaves), max(we for _, we in leaves)
        print(
            f'** Every audit window succeeded, but none had Copilot interaction records ({_fmt_range(first, last)}). '
            f'If people used Copilot in those hours, check: (1) TENANT_ID is the tenant your Copilot users are in; '
            + (f'(2) the app has the Microsoft Graph application permission AuditLogsQuery.Read.All with admin consent '
               f'(the per-service AuditLogsQuery-* permissions do not cover Copilot); '
               if globals().get('AUDIT_SOURCE', 'graph') == 'graph' else
               f'(2) the app has the Office 365 Management APIs application permission ActivityFeed.Read with admin '
               f'consent, and a new Audit.General subscription can take up to 12 hours to publish content'
               + (', and the Microsoft Copilot data connector is connected in Sentinel; '
                  if globals().get('AUDIT_SOURCE') in ('auto', 'sentinel') else '; '))
            + f'(3) auditing is on in Microsoft Purview (Audit); (4) a Purview audit search for the same hours, record '
            f'type CopilotInteraction, finds records. Empty windows are queried once more after they leave the '
            f'look-back (EMPTY_RECHECK_DAYS).'
        )
    return set(keys), rows


def _on_throttle():
    if QUERY_LIMITER.shrink('HTTP 429 (throttled) from Graph', halve=True):
        print(f'    the audit service is throttling (HTTP 429); now {QUERY_LIMITER.limit} audit search(es) at a time')


THROTTLE_LISTENERS[:] = [_on_throttle]
BUDGET_CHECKS[:] = [check_budget]
try:
    with MANIFEST_LOCK:
        manifest = _load_manifest()
    if MODE == 'incremental' and not ALLOW_UNRECOVERED_GAPS:
        assert_no_unrecovered_gaps(end_date - timedelta(days=max(int(BACKFILL_DAYS), 0)), start_date)
    QUERY_LIMITER = AdaptiveLimiter(MAX_CONCURRENT_QUERIES, MIN_CONCURRENT_QUERIES, check=check_budget)
    SOURCE_PLAN.update(plan_audit_sources(windows))
    windows, OUT_OF_REACH, _write_mode = drop_out_of_reach(windows, WRITE_MODE)
    if OUT_OF_REACH:
        print(f'Note: {len(OUT_OF_REACH)} window(s) ({_fmt_range(min(w[0] for w in OUT_OF_REACH), max(w[1] for w in OUT_OF_REACH))}) '
              f'are older than Sentinel and the Management Activity API hold, so this run leaves them out. '
              f'Set GRAPH_FOR_HISTORY = True for one run to load them from Graph.')
    if _write_mode != WRITE_MODE:
        print(f'Note: WRITE_MODE {WRITE_MODE} -> {_write_mode}, so the rows already in {OUTPUT_TABLE} that this run '
              f"can't reload are kept.")
        WRITE_MODE = _write_mode
    windows, EMPTY_RECHECK_KEYS = plan_empty_rechecks(windows)
    if AUDIT_SOURCE in ('auto', 'feed') and SOURCE_PLAN.get('feed_floor') is not None:
        try:
            PREFETCHED_KEYS.update(prefetch_feed(leaf_windows(windows)))
        except Exception as e:
            SOURCE_STATUS['feed'] = 'forbidden' if source_error_status(e) == 'forbidden' else 'error'
            if not GRAPH_FOR_HISTORY:
                raise RuntimeError(
                    f'The Office 365 Management Activity API could not be read ({type(e).__name__}: {e}). Nothing was '
                    f'written to {OUTPUT_TABLE}. Check ActivityFeed.Read admin consent and the source error, then rerun.'
                ) from e
            print(f'WARNING: the Management Activity API could not be read ({type(e).__name__}: {e}); '
                  f'its windows are read from Graph instead.')
            SOURCE_PLAN['feed_floor'] = None
    if AUDIT_SOURCE in ('auto', 'feed') and SOURCE_PLAN.get('feed_floor') is not None:
        SOURCE_STATUS.setdefault('feed', 'ok')
    ACTIVE_WINDOW_KEYS, total = fetch_windows(windows, EMPTY_RECHECK_KEYS)
finally:
    BUDGET_CHECKS.clear()
    write_source_status()
print(f'Prepared {len(ACTIVE_WINDOW_KEYS)} active window(s); staged files will be read only from this run scope. Manifest: {MANIFEST}')


## 6. Read the staged records (distributed)

Load the staged JSON-lines with Spark — a distributed read, not a driver-side
`createDataFrame` from a giant Python list.

In [ ]:
import os
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, ArrayType

_cols = ['RecordId', 'CreationDate', 'RecordType', 'Operation',
         'AuditData', 'SourceRecordKey', 'AssociatedAdminUnits', 'AssociatedAdminUnitsNames']

# Read only the per-window JSONL files that belong to the current window set. This prevents
# historical staging files from being re-appended on later incremental runs.
_files = list_active_stage_files(STAGING_ABS, ACTIVE_WINDOW_KEYS)
if not _files:
    _empty = StructType([StructField(c, StringType()) for c in _cols])
    raw = spark.createDataFrame([], _empty)
else:
    raw = spark.read.json(_files)

raw = raw.persist()
print(f'Raw DataFrame rows: {raw.count():,} (from {len(_files)} active window file(s))')


## 7. Parse `AuditData` JSON

Schema mirrors the existing `Copilot_Audit_Log_Parser.ipynb` — only the fields the PBIT consumes.


In [ ]:
audit_schema = StructType([
    StructField('CreationTime', StringType()),
    StructField('UserId', StringType()),
    StructField('Workload', StringType()),
    StructField('ApplicationName', StringType()),
    StructField('ClientRegion', StringType()),
    StructField('AgentId', StringType()),
    StructField('AgentName', StringType()),
    # Copilot Studio runtime records: AgentPlatform 'CopilotStudio' and
    # PlatformAgentId '<environment>_<bot id>'. They usually carry no Messages.
    StructField('AgentPlatform', StringType()),
    StructField('PlatformAgentId', StringType()),
    StructField('ConversationId', StringType()),
    StructField('AppIdentity', StructType([
        StructField('AppId', StringType()),
        StructField('DisplayName', StringType()),
        StructField('PublisherId', StringType()),
    ])),
    StructField('CopilotEventData', StructType([
        StructField('AppHost', StringType()),
        StructField('ThreadId', StringType()),
        StructField('ConversationId', StringType()),
        StructField('SensitivityLabelId', StringType()),
        StructField('Contexts', ArrayType(StructType([
            StructField('Type', StringType())
        ]))),
        StructField('AISystemPlugin', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('Name', StringType())
        ]))),
        StructField('ModelTransparencyDetails', ArrayType(StructType([
            StructField('ModelName', StringType())
        ]))),
        StructField('AccessedResources', ArrayType(StructType([
            StructField('Type', StringType()),
            StructField('Action', StringType()),
            StructField('SiteUrl', StringType()),
            StructField('SensitivityLabelId', StringType()),
            StructField('_StableKey', StringType()),
            StructField('_StableOrdinal', StringType()),
        ]))),
        StructField('Messages', ArrayType(StructType([
            StructField('Id', StringType()),
            StructField('isPrompt', StringType()),
            StructField('_StableKey', StringType()),
            StructField('_StableOrdinal', StringType()),
        ]))),
    ])),
])

parsed = (raw
    .filter(F.col('AuditData').isNotNull() & (F.length('AuditData') > 10))
    .withColumn('j', F.from_json('AuditData', audit_schema))
    .filter(F.col('j').isNotNull()))

## 8. Flatten + fan out per (prompt × resource)

Mirrors the M-query exactly: drop responses, expand resources. No-prompt Copilot Studio runtime and Cowork autonomous records are kept as one task row per record.


In [ ]:
_empty_message = F.struct(
    F.lit(None).cast(StringType()).alias('Id'),
    F.lit(None).cast(StringType()).alias('isPrompt'),
    F.lit(None).cast(StringType()).alias('_StableKey'),
    F.lit(None).cast('long').alias('_StableOrdinal'),
)
_empty_resource = F.struct(
    F.lit(None).cast(StringType()).alias('Type'),
    F.lit(None).cast(StringType()).alias('Action'),
    F.lit(None).cast(StringType()).alias('SiteUrl'),
    F.lit(None).cast(StringType()).alias('SensitivityLabelId'),
    F.lit('resource:none').cast(StringType()).alias('_StableKey'),
    F.lit(0).cast('long').alias('_StableOrdinal'),
)


def _nonblank(col):
    return F.length(F.trim(F.coalesce(col.cast('string'), F.lit('')))) > 0


# Microsoft documents AppIdentity as a 'workload.appGroup.appName' string (for example
# 'Copilot.Studio.<AppId>' or 'MicrosoftAgent.<name>.P_<id>'). The struct fields above
# stay null for that shape, so keep the raw text for the agent-type rules. An object
# value falls back to its DisplayName.
_app_identity_json = F.get_json_object(F.col('AuditData'), '$.AppIdentity')
_app_identity_text = (
    F.when(F.trim(_app_identity_json).startswith('{'), F.col('j.AppIdentity.DisplayName'))
     .otherwise(_app_identity_json)
)


def _ced_text(field):
    return F.get_json_object(F.col('AuditData'), f'$.CopilotEventData.{field}')


_base = (
    parsed
    .select(
        F.col('RecordId'),
        F.col('Operation'),
        F.col('AuditData').alias('_AuditDataRaw'),
        F.col('SourceRecordKey').alias('_SourceRecordKey'),
        F.col('j.CreationTime').cast('timestamp').alias('CreationDate'),
        F.col('j.AgentId').alias('AgentId'),
        F.col('j.AgentName').alias('AgentName'),
        F.col('j.AgentPlatform').alias('AgentPlatform'),
        F.col('j.PlatformAgentId').alias('PlatformAgentId'),
        F.col('j.AppIdentity.AppId').alias('AppIdentity_AppId'),
        F.col('j.AppIdentity.DisplayName').alias('AppIdentity_DisplayName'),
        F.col('j.AppIdentity.PublisherId').alias('AppIdentity_PublisherId'),
        _app_identity_text.alias('AppIdentity_Text'),
        # Agent fields current exports carry inside CopilotEventData (agent-type rules).
        _ced_text('TargetPlatformAgentId').alias('Agent_TargetPlatformId'),
        F.coalesce(_ced_text('TargetAgentName'), _ced_text('AgentName')).alias('Agent_TargetName'),
        _ced_text('PlatformAgentType').alias('Agent_PlatformType'),
        F.col('j.ApplicationName').alias('ApplicationName'),
        F.col('j.ClientRegion').alias('ClientRegion'),
        F.col('j.UserId').alias('Audit_UserId'),
        F.lower(F.trim(F.col('j.UserId'))).alias('Audit_UserId_Normalized'),
        F.col('j.Workload').alias('Workload'),
        F.col('j.CopilotEventData.AppHost').alias('AppHost'),
        # Runtime records name the conversation ConversationId rather than ThreadId.
        F.coalesce(
            F.when(_nonblank(F.col('j.CopilotEventData.ThreadId')), F.col('j.CopilotEventData.ThreadId')),
            F.when(_nonblank(F.col('j.CopilotEventData.ConversationId')), F.col('j.CopilotEventData.ConversationId')),
            F.when(_nonblank(F.col('j.ConversationId')), F.col('j.ConversationId')),
        ).alias('ThreadId'),
        F.col('j.CopilotEventData.SensitivityLabelId').alias('SensitivityLabelId'),
        F.col('j.CopilotEventData.Contexts')[0]['Type'].alias('Context_Type'),
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Id'].alias('AISystemPlugin_Id'),
        F.col('j.CopilotEventData.AISystemPlugin')[0]['Name'].alias('AISystemPlugin_Name'),
        F.col('j.CopilotEventData.ModelTransparencyDetails')[0]['ModelName']
            .alias('ModelTransparencyDetails_ModelName'),
        F.col('j.CopilotEventData.AccessedResources').alias('Resources'),
        F.col('j.CopilotEventData.Messages').alias('Messages'),
        F.size('j.CopilotEventData.AccessedResources').alias('Resource_Count'),
    )
)

_ambiguous_synthetic_records = (
    _base
    .filter(~_nonblank(F.col('RecordId')) & _nonblank(F.col('_SourceRecordKey')))
    .groupBy('_SourceRecordKey')
    .count()
    .filter(F.col('count') > 1)
    .limit(1)
    .collect()
)
if _ambiguous_synthetic_records:
    raise RuntimeError(
        "Missing RecordId produced ambiguous synthetic SourceRecordKey values; refusing unsafe dedup."
    )

# A record with no prompt message has no prompt to count. Copilot Studio runtime
# records (agents used in Teams, websites and other channels) and Microsoft 365
# Copilot Cowork scheduled/autonomous runs are the exceptions: they are the only
# audit trail of that usage, so they are kept as one task row per record with
# Prompts_Available = 'FALSE'. Other no-prompt records are still dropped.
_has_messages = F.coalesce(F.size('Messages'), F.lit(-1)) > 0
_has_prompt_messages = F.coalesce(
    F.expr("exists(Messages, m -> lower(cast(m.isPrompt as string)) = 'true')"),
    F.lit(False),
)
_copilot_studio_runtime = (
    F.lower(F.regexp_replace(F.coalesce(F.col('AgentPlatform'), F.lit('')), '\\s', '')) == 'copilotstudio'
)


def is_cowork_autonomous(app_host_col, app_identity_col):
    _app_host = F.lower(F.trim(F.coalesce(app_host_col.cast('string'), F.lit(''))))
    _app_identity = F.lower(F.trim(F.coalesce(app_identity_col.cast('string'), F.lit(''))))
    return (_app_host == 'cowork') | _app_identity.startswith('copilot.m365copilot.cowork')


_cowork_autonomous = is_cowork_autonomous(F.col('AppHost'), F.col('AppIdentity_Text'))
_task_row_record = _copilot_studio_runtime | _cowork_autonomous

flat = (
    _base
    .withColumn('Prompts_Available', F.when(_has_prompt_messages, F.lit('TRUE')).otherwise(F.lit('FALSE')))
    .withColumn('_Task_Row_Placeholder', (F.col('Prompts_Available') == 'FALSE') & _task_row_record)
    .withColumn(
        'Source_RecordKey',
        F.when(_nonblank(F.col('_SourceRecordKey')), F.trim(F.col('_SourceRecordKey').cast('string')))
         .when(_nonblank(F.col('RecordId')), F.concat(F.lit('rid:'), F.trim(F.col('RecordId').cast('string'))))
    )
    .select(
        '*',
        F.posexplode_outer(
            F.when(F.col('_Task_Row_Placeholder'), F.array(_empty_message))
             .when(F.size('Messages') > 0, F.col('Messages'))
             .otherwise(F.array(_empty_message))
        ).alias('Message_ArrayOrdinal', 'msg'),
    )
    .filter(
        (F.lower(F.col('msg.isPrompt').cast('string')) == 'true')
        | F.col('_Task_Row_Placeholder')
    )
    .select(
        '*',
        F.posexplode_outer(
            F.when(F.col('_Task_Row_Placeholder'), F.array(_empty_resource))
             .when(F.size('Resources') > 0, F.col('Resources'))
             .otherwise(F.array(_empty_resource))
        ).alias('Resource_ArrayOrdinal', 'res'),
    )
    .withColumn(
        'Source_MessageKey',
        F.coalesce(
            F.trim(F.col('msg._StableKey').cast('string')),
            F.when(F.col('_Task_Row_Placeholder'), F.lit('message:none')),
        ),
    )
    .withColumn('Source_ResourceKey', F.trim(F.col('res._StableKey').cast('string')))
    .withColumn(
        'Message_Ordinal',
        F.coalesce(F.col('msg._StableOrdinal').cast('long'), F.col('Message_ArrayOrdinal').cast('long')),
    )
    .withColumn(
        'Resource_Ordinal',
        F.coalesce(F.col('res._StableOrdinal').cast('long'), F.col('Resource_ArrayOrdinal').cast('long'), F.lit(0).cast('long')),
    )
)

_missing_identity_keys = (
    flat
    .filter(
        ~_nonblank(F.col('Source_RecordKey'))
        | ~_nonblank(F.col('Source_MessageKey'))
        | ~_nonblank(F.col('Source_ResourceKey'))
    )
    .limit(1)
    .collect()
)
if _missing_identity_keys:
    raise RuntimeError(
        "Canonical audit identity fields are missing from staged data; rerun ingestion with the canonical stage helpers."
    )

flat = (
    flat
    .withColumn(
        'Id',
        F.sha2(
            F.concat_ws(
                '||',
                F.col('Source_RecordKey'),
                F.col('Source_MessageKey'),
                F.col('Source_ResourceKey'),
            ),
            256,
        ),
    )
    .select(
        'Id', 'RecordId', 'Source_RecordKey', 'Source_MessageKey', 'Source_ResourceKey',
        'CreationDate', 'AgentId', 'AgentName', 'AgentPlatform', 'PlatformAgentId',
        'AppIdentity_AppId', 'AppIdentity_DisplayName', 'AppIdentity_PublisherId',
        'AppIdentity_Text', 'Agent_TargetPlatformId', 'Agent_TargetName', 'Agent_PlatformType',
        'ApplicationName', 'ClientRegion',
        'Audit_UserId', 'Audit_UserId_Normalized', 'Workload',
        'AppHost', 'ThreadId', 'SensitivityLabelId',
        'Context_Type',
        'AISystemPlugin_Id', 'AISystemPlugin_Name',
        'ModelTransparencyDetails_ModelName',
        F.col('res.Type').alias('AccessedResource_Type'),
        F.col('res.Action').alias('AccessedResource_Action'),
        F.col('res.SiteUrl').alias('AccessedResource_SiteUrl'),
        F.col('res.SensitivityLabelId').alias('AccessedResource_SensitivityLabelId'),
        F.when(F.col('_Task_Row_Placeholder'), F.lit('message:none'))
         .otherwise(F.col('msg.Id')).alias('Message_Id'),
        F.when(F.col('_Task_Row_Placeholder'), F.lit('false'))
         .otherwise(F.col('msg.isPrompt')).alias('Message_isPrompt'),
        'Prompts_Available',
        F.col('Message_Ordinal').cast('long').alias('Message_Ordinal'),
        F.col('Resource_Ordinal').cast('long').alias('Resource_Ordinal'),
        F.coalesce(F.col('Resource_Count'), F.lit(1)).cast('long').alias('Resource_Count'),
        F.to_date('CreationDate').alias('InteractionDate'),
        F.expr("date_trunc('week', CreationDate)").cast('date').alias('WeekStart'),
        F.expr("date_trunc('month', CreationDate)").cast('date').alias('MonthStart'),
    )
    .dropDuplicates(['Id'])
)

## 9. Derive `Agent_TitleID` + `Agent_EntraId`

`Agent_TitleID` keeps the legacy Copilot Studio logic (existing parser + M-query).
`Agent_EntraId` captures the **Microsoft Entra Agent ID** GUID that Agent 365 now
stamps into the audit `AgentId` (instead of the `CopilotStudio.Declarative.{title}`
string). Without it, Entra-registered agents fall through to NULL and drop out of
the Agents join. Both keys are emitted so old and new agents resolve via the
registry crosswalk; neither overwrites the other.


In [ ]:
flat = flat.withColumn('Agent_TitleID', F.expr(r'''
    CASE
      WHEN AgentId IS NULL THEN NULL
      WHEN AgentId LIKE '%CopilotStudio.Declarative.%' THEN
           split(split(AgentId, 'CopilotStudio\\.Declarative\\.')[1], '\\.')[0]
      WHEN AgentId LIKE 'P\\_%' OR AgentId LIKE 'T\\_%' THEN
           split(AgentId, '\\.')[0]
      ELSE NULL
    END
'''))

# Microsoft Entra Agent ID (Agent 365): the audit AgentId now carries an Entra
# Agent ID GUID rather than the CopilotStudio.Declarative.{title} string, so the
# legacy CASE above lands NULL. Capture the GUID only when no legacy Title ID was
# parsed, so a GUID embedded inside a declarative title is never mistaken for it.
flat = flat.withColumn('Agent_EntraId', F.expr(r'''
    CASE
      WHEN Agent_TitleID IS NOT NULL THEN NULL
      WHEN AgentId IS NULL THEN NULL
      ELSE nullif(
             regexp_extract(
               AgentId,
               '[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}',
               0),
             '')
    END
'''))
# An all-zero AgentId (Fabric multi-agent orchestration) is not an agent identity.
flat = flat.withColumn(
    'Agent_EntraId',
    F.when(F.regexp_replace(F.coalesce(F.col('Agent_EntraId'), F.lit('')), '[-0]', '') == '', F.lit(None).cast(StringType()))
     .otherwise(F.col('Agent_EntraId')),
)

# Copilot Studio runtime records carry the bot, not the Title ID:
# PlatformAgentId = '<environment id>_<bot id>'. A bare GUID is taken as the bot id.
PLATFORM_AGENT_ID_PATTERN = r'^(.*)_([0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12})$'
BARE_GUID_PATTERN = r'^[0-9a-fA-F]{8}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{4}-[0-9a-fA-F]{12}$'
_platform_agent_id = F.trim(F.coalesce(F.col('PlatformAgentId'), F.lit('')))
flat = (flat
    .withColumn('Agent_BotId', F.lower(
        F.when(_platform_agent_id.rlike(PLATFORM_AGENT_ID_PATTERN),
               F.regexp_extract(_platform_agent_id, PLATFORM_AGENT_ID_PATTERN, 2))
         .when(_platform_agent_id.rlike(BARE_GUID_PATTERN), _platform_agent_id)))
    .withColumn('Agent_EnvironmentId',
        F.when(_platform_agent_id.rlike(PLATFORM_AGENT_ID_PATTERN),
               F.regexp_extract(_platform_agent_id, PLATFORM_AGENT_ID_PATTERN, 1))))

# Records that are not end-user agent usage. Flagged with the first matching reason:
#   AppHost "Copilot Studio"       -> the maker's test pane
#   AppHost "pva-maker-evaluation" -> Copilot Studio evaluation runs
#   AppHost "agentic-builder"      -> agent authoring in Agent Builder
#   AppHost "autonomous"           -> autonomous (trigger-driven) agent runs
#   AppHost "workflow-agents"      -> workflow agent runs
#   no messages + AppHost "M365Copilot" -> the Copilot Studio runtime twin of an
#       M365 Copilot chat that is already counted from its own prompt record
#   all-zero AgentId               -> Fabric multi-agent orchestration
# Plain SQL (lower/trim/coalesce/replace) so the same text runs in Spark and in the tests.
EXCLUDE_REASON_SQL = """
CASE
  WHEN lower(trim(coalesce(AppHost, ''))) = 'copilot studio' THEN 'Copilot Studio test pane'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'pva-maker-evaluation' THEN 'Maker evaluation'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'agentic-builder' THEN 'Agent authoring'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'autonomous' THEN 'Autonomous run'
  WHEN lower(trim(coalesce(AppHost, ''))) = 'workflow-agents' THEN 'Workflow run'
  WHEN upper(trim(coalesce(Prompts_Available, 'TRUE'))) = 'FALSE'
       AND lower(trim(coalesce(AppHost, ''))) = 'm365copilot' THEN 'M365 Copilot twin'
  WHEN trim(coalesce(AgentId, '')) <> ''
       AND replace(replace(lower(trim(coalesce(AgentId, ''))), '-', ''), '0', '') = ''
       THEN 'Fabric multi-agent'
  ELSE NULL
END
"""

# Flag only; the processor drops the reasons listed in its DROP_EXCLUDE_REASONS.
flat = flat.withColumn('Exclude_Reason', F.expr(EXCLUDE_REASON_SQL))

## 10. Write to Lakehouse Delta table

- **Incremental** (`MODE='incremental'`) merges the batch by `Id`, reading only the months the batch touches.
- **Backfill** (`MODE='backfill'`) replaces rows from the earliest `CreationDate` it read onwards and keeps
  older rows, so a reload never shortens the history. Rows without a `CreationDate` are replaced by every backfill.
- A backfill that returns no records leaves the table as it is. If the table doesn't exist yet, it's created.
- A table from before the stable key columns is replaced whole, once, by the next backfill.
- New columns are added to the table. A changed column type stops the write: drop the table, then run a
  backfill of the history you need.


In [ ]:
# Backfill write helpers. A backfill replaces parsed rows only from the earliest CreationDate it read
# onwards, so the history before its window is kept: a reload never shortens the table.
from datetime import datetime, timedelta, timezone

import pyspark.sql.functions as F

PARTITION_COLUMN = 'MonthStart'
_EPOCH = datetime(1970, 1, 1, tzinfo=timezone.utc)


def utc_timestamp_literal(micros):
    """Spark SQL literal for a UTC instant given as whole microseconds since the epoch."""
    if isinstance(micros, bool) or not isinstance(micros, int):
        raise ValueError(f'expected whole microseconds since the epoch, got {micros!r}')
    instant = (_EPOCH + timedelta(microseconds=micros)).replace(tzinfo=None)
    return f"TIMESTAMP '{instant.isoformat(sep=' ', timespec='microseconds')}Z'"


def backfill_replace_predicate(earliest, column='CreationDate'):
    """replaceWhere predicate for a backfill whose earliest record is `earliest` (microseconds since
    the epoch, or None when no record has a date). Rows without a date can't be placed in time, so
    every backfill replaces them."""
    if earliest is None:
        return f'`{column}` IS NULL'
    return f'`{column}` >= {utc_timestamp_literal(earliest)} OR `{column}` IS NULL'


def plan_backfill_write(table_exists, batch_rows, earliest, missing_keys=(), column='CreationDate'):
    """(action, replaceWhere predicate or None) for a backfill batch.

    create:    no table yet, so the batch becomes the table.
    rebuild:   the table predates the stable key columns, so it's replaced whole, once.
    unchanged: the backfill returned no records. An empty write with replaceWhere would delete the
               range, so nothing is written.
    replace:   rows from the batch's earliest record onwards are replaced; older rows are kept."""
    if not table_exists:
        return 'create', None
    if missing_keys:
        return 'rebuild', None
    if not batch_rows:
        return 'unchanged', None
    return 'replace', backfill_replace_predicate(earliest, column)


def batch_extent(df, column='CreationDate'):
    """(row count, earliest `column` in microseconds since the epoch, or None), in one pass."""
    row = df.agg(F.count(F.lit(1)).alias('rows'),
                 F.expr(f'unix_micros(min(`{column}`))').alias('earliest')).collect()[0]
    earliest = row['earliest']
    return int(row['rows']), None if earliest is None else int(earliest)


def _table_partition_columns(table_name):
    return list(spark.sql(f'DESCRIBE DETAIL {table_name}').collect()[0]['partitionColumns'] or [])


def ensure_month_partitions(table_name, partition_column=PARTITION_COLUMN):
    """Store the table by month, once, so later writes rewrite only the months they touch.
    Returns False when the table has no `partition_column` to store it by."""
    if partition_column not in spark.table(table_name).columns:
        return False
    if _table_partition_columns(table_name) != [partition_column]:
        print(f'migrating {table_name} to month partitions ({partition_column}), one time...')
        (spark.table(table_name).write
            .format('delta')
            .mode('overwrite')
            .option('overwriteSchema', 'true')
            .partitionBy(partition_column)
            .saveAsTable(table_name))
    return True


def write_backfill(df, table_name, key_columns=(), partition_column=PARTITION_COLUMN, column='CreationDate'):
    """Write a backfill batch and return the action taken (see plan_backfill_write)."""
    exists = spark.catalog.tableExists(table_name)
    missing = []
    if exists:
        target_columns = spark.table(table_name).columns
        missing = [c for c in key_columns if c not in target_columns]
    rows, earliest = batch_extent(df, column)
    action, predicate = plan_backfill_write(exists, rows, earliest, missing, column)
    if action == 'unchanged':
        print(f'The backfill returned no records, so {table_name} is left as it is.')
        return action
    if action == 'replace':
        ensure_month_partitions(table_name, partition_column)
        # Delta checks each written row against replaceWhere and fails the write if one doesn't match,
        # and every batch row does: it's on or after the earliest date, or has none. The predicate is on
        # the record's date, never the month column, so the older rows of the first month are kept.
        # overwriteSchema doesn't apply with replaceWhere; mergeSchema adds new columns.
        (df.write
            .format('delta')
            .mode('overwrite')
            .option('mergeSchema', 'true')
            .option('replaceWhere', predicate)
            .saveAsTable(table_name))
        print(f'{table_name}: replaced rows where {predicate}; older rows kept.')
        return action
    if action == 'rebuild':
        print(f'{table_name} has no stable key columns {missing}, so this backfill replaces it whole, once.')
    writer = df.write.format('delta').mode('overwrite').option('overwriteSchema', 'true')
    if partition_column in df.columns:
        writer = writer.partitionBy(partition_column)
    writer.saveAsTable(table_name)
    return action

In [ ]:
def _validate_batch_keys(df, key_columns, dataset_name: str):
    missing = [c for c in key_columns if c not in df.columns]
    if missing:
        raise RuntimeError(
            f'{dataset_name} is missing stable key columns {missing}; cannot perform idempotent incremental merge.'
        )
    has_blank = (
        df.where(F.col('Id').isNull() | (F.trim(F.col('Id').cast('string')) == ''))
          .limit(1)
          .count()
    )
    if has_blank:
        raise RuntimeError(
            f'{dataset_name} contains blank Id values; rerun after fixing key extraction rather than appending duplicates.'
        )

def month_bounded_merge_condition(batch_months, partition_column='MonthStart'):
    # MERGE condition that only scans the target months present in the batch. A row's Id
    # is derived from its audit record, so a re-delivered row always lands in the same month.
    dates = sorted({str(m)[:10] for m in batch_months if m is not None})
    bounds = []
    if dates:
        bounds.append(f"t.`{partition_column}` IN ({', '.join(f'DATE{chr(39)}{d}{chr(39)}' for d in dates)})")
    if any(m is None for m in batch_months):
        bounds.append(f't.`{partition_column}` IS NULL')
    if not bounds:
        return 't.`Id` = s.`Id`'
    return f"t.`Id` = s.`Id` AND ({' OR '.join(bounds)})"


def write_output(df):
    if WRITE_MODE not in ('replace', 'merge'):
        raise ValueError(f'Unsupported WRITE_MODE={WRITE_MODE!r}')
    if WRITE_MODE == 'merge':
        _validate_batch_keys(df, KEY_COLUMNS, 'current parsed batch')
        if spark.catalog.tableExists(OUTPUT_TABLE):
            target_columns = spark.table(OUTPUT_TABLE).columns
            validate_incremental_target_columns(target_columns, KEY_COLUMNS, OUTPUT_TABLE)
            from delta.tables import DeltaTable
            # New columns (e.g. Agent_BotId, Exclude_Reason) reach an existing table only
            # with schema auto-merge; otherwise updateAll/insertAll silently drop them.
            spark.conf.set('spark.databricks.delta.schema.autoMerge.enabled', 'true')
            bounded = PARTITION_COLUMN in target_columns and PARTITION_COLUMN in df.columns
            if bounded:
                # One-time migration: store the table by month so each MERGE only
                # rewrites the months it touches instead of scanning the whole history.
                ensure_month_partitions(OUTPUT_TABLE)
            batch_months = ([r[0] for r in df.select(PARTITION_COLUMN).distinct().collect()]
                            if bounded else [])
            cond = month_bounded_merge_condition(batch_months, PARTITION_COLUMN) if bounded else 't.`Id` = s.`Id`'
            tgt = DeltaTable.forName(spark, OUTPUT_TABLE)
            (tgt.alias('t').merge(df.alias('s'), cond)
                .whenMatchedUpdateAll()
                .whenNotMatchedInsertAll()
                .execute())
            return 'merge'
    # A backfill, or an incremental run with no table yet: create the table, or replace it from
    # the batch's earliest record onwards.
    return write_backfill(df, OUTPUT_TABLE, KEY_COLUMNS)

write_strategy = write_output(flat)
clear_recheck_pending()
row_count = spark.table(OUTPUT_TABLE).count()
print(f'✓ Rows in {OUTPUT_TABLE}: {row_count:,} (strategy={write_strategy})')

## 11. Verify

Spot-check the output. Expect populated `AISystemPlugin_Id` (e.g. `BingWebSearch`), `Workload`, `AppHost`.


In [ ]:
spark.table(OUTPUT_TABLE).select(
    'AISystemPlugin_Id', 'Workload', 'AppHost'
).groupBy('AISystemPlugin_Id', 'Workload', 'AppHost').count().orderBy(F.desc('count')).show(20, truncate=False)


---
**Connect the PBIT**: open the Fabric variant of either dashboard and supply the **Fabric SQL Endpoint** + **Lakehouse Database** parameters as before. The PBIT reads `Copilot_Interactions_Parsed` directly via the SQL endpoint — no other parameter changes needed when switching from the CSV-based parser to this direct ingester.
